# Análisis de convergencia de soluciones de EDP mediante redes neuronales físicamente informadas
## Ecuación de Poisson en una y dos dimensiones

Pontificia Universidad Católica del Perú · Maestría en Matemática Aplicada

El estudio numérico se organiza en tres partes. La primera introduce el entrenamiento de redes neuronales mediante regresión, una ecuación diferencial ordinaria y problemas de Poisson con y sin observaciones interiores. La segunda compara distintas pérdidas regularizadas y reproduce los experimentos elípticos de Shin, Darbon y Karniadakis (2020). La tercera examina el control Hölder del residuo y contrasta el método PINN con discretizaciones por elementos y volúmenes finitos para Poisson 2D.

Las soluciones se representan en escala lineal, las componentes de la función de pérdida en escala semilogarítmica y las comparaciones de error frente al refinamiento en escala log-log. Las figuras y las tablas numéricas se obtienen de la ejecución de los experimentos.


## Ejecución y organización de los cálculos

Cada experimento declara sus parámetros, su red neuronal, la ecuación que se resuelve y las funciones necesarias para entrenar y evaluar la solución. Se pueden ejecutar los apartados individualmente, respetando el orden de sus celdas. Las figuras y los archivos CSV se guardan en Google Drive cuando está habilitada la opción `GUARDAR_EN_DRIVE` de cada apartado.

`MODO = 'experimento'` utiliza el número de épocas y los tamaños de muestra del estudio. Para comprobar que un apartado funciona antes de iniciar el entrenamiento completo puede usarse `MODO = 'demostracion'`; esas salidas breves no deben interpretarse como resultados de convergencia.

En la reproducción de Shin hay una configuración adicional, `SHIN_PROTOCOLO`, que distingue una prueba acotada del barrido basado en la sección 4.1 del artículo. Los entrenamientos prolongados se guardan por etapas para recuperar su estado tras una interrupción.

La última sección reúne las imágenes por sus nombres de archivo. Es la única parte que presupone la existencia de resultados de varios experimentos.


In [ ]:
# Información del entorno; los experimentos no dependen de esta celda.
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))

### Métodos numéricos para ecuaciones diferenciales

- Los problemas simples pueden resolverse analíticamente.
- Por ejemplo, considere la velocidad:

$$v(t) = \frac{d x}{d t} = \lim_{h \rightarrow 0} \frac{x(t+h) - x(t)}{h}$$

<img src="https://github.com/FilippoMB/Physics-Informed-Neural-Networks-tutorial/blob/main/figs/simple_pde.png?raw=1" style="width: 40%; display: block; margin: auto;">

- Solución:

$$
v(t) =
\begin{cases}
3/2 & \text{if}\; t \in \{ 0, 2 \} \\
0 & \text{if}\; t \in \{ 2, 4 \} \\
-1/3 & \text{if}\; t \in \{ 4, 7 \}
\end{cases}
$$

<img src="https://github.com/FilippoMB/Physics-Informed-Neural-Networks-tutorial/blob/main/figs/numerical_pde.png?raw=1" style="width: 40%; display: block; margin: auto;">

- En numerosos problemas de interés aplicado no se dispone de una solución analítica explícita.
- En esos casos se emplean métodos numéricos para aproximar las soluciones.
- Por ejemplo, aplican la definición de derivada para todos los puntos del dominio del tiempo.

**Aspectos a considerar en los métodos de discretización**

- La precisión y el costo dependen del tamaño de malla, del orden del método y de la geometría del dominio.
- En geometrías complejas puede ser necesario construir mallas adecuadas.
- La incorporación de observaciones y la solución de problemas inversos exige formular problemas adicionales.
- Para Poisson 1D y 2D, los métodos clásicos constituyen una referencia numérica apropiada frente a las PINNs.

## Redes Neuronales (NN)

<img src="https://github.com/FilippoMB/Physics-Informed-Neural-Networks-tutorial/blob/main/figs/neural_net.png?raw=1" style="width: 40%; display: block; margin: auto;">

- Bajo hipótesis sobre las activaciones y la arquitectura, ciertas familias de redes son densas en espacios de funciones continuas sobre compactos.
- Permiten aproximar relaciones a partir de observaciones $\boldsymbol{X}$.
- Son entrenados para minimizar una pérdida, por ejemplo, el error entre las predicciones $\boldsymbol{\hat{y}}$ y los resultados deseados $\boldsymbol{y}$.

Comenzamos creando una red neuronal simple usando **PyTorch**


In [ ]:
# Regresión simple
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {'ep_regresion': 150 if MODO == 'demostracion' else 1000}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Regresión simple:', DEVICE, '| salida:', SALIDA)

#Librerías

import torch
from torch import nn
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib import cm

In [ ]:
# Definimos una red neuronal simple para la regresión

class simple_NN(nn.Module):
    def __init__(self):
        super(simple_NN, self).__init__()
        self.linear_tanh_stack = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        out = self.linear_tanh_stack(x)
        return out

Entonces,

- Creamos un pequeño conjunto de datos ${x_i, y_i}_{i=1, \dots 5}$.
- Usar la NN para hacer predicciones: $\hat{y}_i = \rm{NN}(x_i)$.
- Entrenamos la NN minimizando $\rm{MSE}(\boldsymbol{y}, \boldsymbol{\hat{y}})$.

In [ ]:
# Definimos la base de datos

x_train = torch.tensor([[1.1437e-04],
        [1.4676e-01],
        [3.0233e-01],
        [4.1702e-01],
        [7.2032e-01]], dtype=torch.float32)
y_train = torch.tensor([[1.0000],
        [1.0141],
        [1.0456],
        [1.0753],
        [1.1565]], dtype=torch.float32)

In [ ]:
# Graficar
plt.figure(figsize=(8, 6))
plt.scatter(x_train, y_train, color='blue', label='Datos de entrenamiento')
plt.xlim(0, 1.5)
plt.ylim(min(y_train)-0.1, max(y_train)+0.1)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Gráfica de los datos de entrenamiento')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Inicializamos el modelo
model = simple_NN()

# definimos la función de pérdida y el optimizador
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

# Guardamos la pérdida para estudiar el aprendizaje en escala semilogarítmica.
historial_regresion = []

# Entrenamiento
for ep in range(CFG['ep_regresion']):

    # Calcula el error
    pred = model(x_train)
    loss = loss_fn(pred, y_train)

    # Guardamos la pérdida SIN transformarla a logaritmos.
    historial_regresion.append({'epoca': ep + 1, 'perdida': float(loss.detach())})

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if ep % 200 == 0:
        print(f"epoch: {ep}, loss: {loss.item():>7f}")
# CSV con los valores reales de la función de pérdida.
guardar_csv('historial_regresion.csv', historial_regresion)


In [ ]:
# evaluamos el modelo en todos los data points del dominio
domain = [0.0, 1.5]
x_eval = torch.linspace(domain[0], domain[1], steps=100).reshape(-1, 1)
with torch.no_grad():
    f_eval = model(x_eval)

# plotting
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(x_train.detach().numpy(), y_train.detach().numpy(), label="Training data", color="blue")
ax.plot(x_eval.detach().numpy(), f_eval.detach().numpy(), label="NN approximation", color="black")
ax.set(title="Neural Network Regression", xlabel="$x$", ylabel="$y$")
ax.legend();

# Guardar la aproximación de regresión.
fig.savefig(SALIDA/'regresion_simple.png', dpi=180, bbox_inches='tight')
plt.show()

**Pérdida de entrenamiento de la regresión.** La solución aproximada se muestra en escala lineal; para visualizar la reducción del error cuadrático medio durante las épocas utilizamos una escala **semilogarítmica**, con eje horizontal lineal y eje vertical logarítmico.

In [ ]:
# ----------------------------------------------------------
# Evolución del error cuadrático medio durante la regresión
# ----------------------------------------------------------
# semilogy cambia únicamente la escala de visualización.
# Los valores guardados en el CSV siguen siendo los errores originales.
epocas = [fila['epoca'] for fila in historial_regresion]
perdidas = [fila['perdida'] for fila in historial_regresion]

plt.figure(figsize=(8, 4))
plt.semilogy(epocas, perdidas, label='MSE de entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Error cuadrático medio (MSE)')
plt.title('Regresión: evolución de la función de pérdida')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
guardar_figura('perdida_regresion.png')

- La NN hace un buen trabajo ajustando las muestras de datos.
- Sin embargo, no tiene información sobre qué función debe aprender cuando $x>0.8$.

## Redes neuronales físicamente informadas (PINNs)

- Utiliza EDPs para ajustar la salida de la NN.
- Entrena el modelo con una pérdida adicional que penalice la violación de la EDP.

$$ \mathcal{L}_{\text{tot}} =  \mathcal{L}_{\text{data}} + \mathcal{L}_{\text{PDE}}$$


<img src="https://github.com/FilippoMB/Physics-Informed-Neural-Networks-tutorial/blob/main/figs/neural_net_pde.png?raw=1" style="width: 50%; display: block; margin: auto;">

**Alcance de la formulación PINN**

Una PINN combina observaciones con restricciones expresadas por ecuaciones diferenciales y condiciones auxiliares. Esta formulación permite entrenar sin disponer necesariamente de valores observados de la solución en todos los puntos interiores.

La pérdida residual pequeña no implica, por sí sola, un error pequeño respecto de la solución exacta. El paso de la pérdida empírica al error funcional requiere hipótesis de muestreo, regularidad y estabilidad, que se estudian en el capítulo III. Tampoco se presupone que una PINN sea más eficiente que FEM o FVM; esa comparación se examina al final del cuaderno.

### Ejemplo I: crecimiento poblacional

Ecuación logística para modelar el crecimiento poblacional:

$$ \frac{d f(t)}{d t} = Rf(t)(1-f(t))$$

- $f(t)$ es el crecimiento poblacional a lo largo del tiempo $t$
- $R$ es la tasa máxima de crecimiento
- Para identificar una solución, se debe imponer una condición inicial, por ejemplo, en $t=0$:

$$f(t=0)=0.5$$

In [ ]:
# Ecuación logística
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {'ep_logistica': 200 if MODO == 'demostracion' else 2000}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Ecuación logística:', DEVICE, '| salida:', SALIDA)

# Red neuronal declarada localmente: NO depende de regresión.
# Definimos una red neuronal simple para la regresión

class simple_NN(nn.Module):
    def __init__(self):
        super(simple_NN, self).__init__()
        self.linear_tanh_stack = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        out = self.linear_tanh_stack(x)
        return out
# Dominio y cuadrícula de evaluación propios de la ecuación logística.
domain = [0.0, 1.5]
x_eval = torch.linspace(*domain, steps=100).reshape(-1, 1)

#Definimos el valor de la tasa e indicamos el valor que toma f en la condición inicial.

R = 1.0
ft0 = 0.5

- Utilizamos la NN para modelar $f(t)$, es decir, $f(t) = \rm{NN}(t)$.
- Podemos calcular fácilmente la derivada $\frac{d\rm{NN}(t)}{dt}$ gracias a la diferenciación automática proporcionada por las bibliotecas de aprendizaje profundo.

In [ ]:
def df(f: simple_NN, x: torch.Tensor = None, order: int = 1) -> torch.Tensor:
    """Calcula la derivada de la red neuronal con respecto a las características de entrada usando el motor autograd de PyTorch"""
    df_value = f(x)
    for _ in range(order):
        df_value = torch.autograd.grad(
            df_value,
            x,
            grad_outputs=torch.ones_like(x),
            create_graph=True,
            retain_graph=True,
        )[0]

    return df_value

- Queremos que nuestra NN satisfaga la siguiente ecuación:

$$ \frac{d\rm{NN}(t)}{dt} - R\rm{NN}(t)(1-\rm{NN}(t)) = 0 $$

- Para lograr esto, agregamos el siguiente término de regularización físicamente informado a la pérdida:

$$ \mathcal{L}_\rm{PDE} = \frac{1}{N} \sum_{i=1}^N \left( \frac{d\rm{NN}}{dt} \bigg\rvert_{t_i} - R \rm{NN}(t_i) (1-\rm{NN}(t_i)) \right)^2 $$

- donde $t_i$ son **puntos de colocación**, es decir, un conjunto de puntos del dominio donde evaluamos la ecuación diferencial.

In [ ]:
#Generar 10 puntos de colocación distribuidos uniformemente

t = torch.linspace(domain[0], domain[1], steps=10, requires_grad=True).reshape(-1, 1)

- Minimizar solo $\mathcal{L}_\rm{PDE}$ no garantiza una solución única.
- Debemos incluir la condición de frontera añadiendo la siguiente pérdida:

$$ \mathcal{L}_\rm{BC} = \left( \rm{NN}(t_0) - 0.5 \right)^2 $$

- Esto permite que la NN converja a la solución deseada entre las infinitas posibles.

La función de pérdida finalmente está dada por:

$$ \mathcal{L}_\rm{PDE} + \mathcal{L}_\rm{BC} + \mathcal{L}_\rm{data} $$

In [ ]:
# Envuelve todo en una función
def compute_loss(nn: simple_NN,
                 t: torch.Tensor = None,
                 x: torch.Tensor = None,
                 y: torch.Tensor = None,
                 devolver_componentes: bool = False) -> torch.float:
    """Calcula la función de pérdida completa como pérdida de la EDP + pérdida de la condición de frontera
Esta función de pérdida está completamente definida con tensores diferenciables, por lo tanto,
se puede aplicar el método .backward() a ella
    """

    pde_loss = df(nn, t) - R * nn(t) * (1 - nn(t))
    pde_loss = pde_loss.pow(2).mean()

    boundary = torch.Tensor([0.0])
    boundary.requires_grad = True
    bc_loss = nn(boundary) - ft0
    bc_loss = bc_loss.pow(2)

    mse_loss = torch.nn.MSELoss()(nn(x), y)

    tot_loss = pde_loss + bc_loss + mse_loss

    if devolver_componentes:
        return tot_loss, pde_loss, bc_loss.mean(), mse_loss
    return tot_loss

In [ ]:
# Definimos la base de datos

x_train = torch.tensor([[1.1437e-04],
        [1.4676e-01],
        [3.0233e-01],
        [4.1702e-01],
        [7.2032e-01]], dtype=torch.float32)
y_train = torch.tensor([[0.49769],
    [0.54271],
    [0.58026],
    [0.60909],
    [0.67524]], dtype=torch.float32)

In [ ]:
model = simple_NN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

# Histórico del error para graficar el descenso de la pérdida.
historial_logistica = []

# Entrenamiento
for ep in range(CFG['ep_logistica']):

    loss, j_pde, j_frontera, j_datos = compute_loss(
        model, t, x_train, y_train, devolver_componentes=True)

    # Cada componente se calcula en el mismo paso, antes de actualizar la red.
    historial_logistica.append({
        'epoca': ep + 1, 'perdida': float(loss.detach()),
        'pde': float(j_pde.detach()), 'bc': float(j_frontera.detach()),
        'datos': float(j_datos.detach())})

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if ep % 200 == 0:
        print(f"epoch: {ep}, loss: {loss.item():>7f}")
# Guardamos los datos originales, no sus logaritmos.
guardar_csv('historial_logistica.csv', historial_logistica)


In [ ]:
# solución numérica
def logistic_eq_fn(t, f):
    return R * f * (1 - f)

numeric_solution = solve_ivp(
    logistic_eq_fn, domain, [ft0], t_eval=x_eval.squeeze().detach().numpy()
)

f_colloc = solve_ivp(
    logistic_eq_fn, domain, [ft0], t_eval=t.squeeze().detach().numpy()
).y.T

In [ ]:
# evaluación del modelo en el dominio [0, 1.5]
with torch.no_grad():
    f_eval = model(x_eval)

# plotting
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(t.detach().numpy(), f_colloc, label="Puntos de colocación", color="magenta", alpha=0.75)
ax.scatter(x_train.detach().numpy(), y_train.detach().numpy(), label="Datos observados", color="blue")
ax.plot(x_eval.detach().numpy(), f_eval.detach().numpy(), label="solución NN", color="black")
ax.plot(x_eval.detach().numpy(), numeric_solution.y.T,
        label="solución numérica de referencia", color="magenta", alpha=0.75)
ax.set(title="La ecuación logística resuelta con NNs", xlabel="t", ylabel="f(t)")
ax.legend();

# Guardar la comparación con la solución de la EDO.
fig.savefig(SALIDA/'logistica_edo.png', dpi=180, bbox_inches='tight')
plt.show()

**Pérdida de la PINN para la ecuación logística.** La comparación entre la solución neuronal y la solución numérica permanece en escala lineal. Graficamos la pérdida total en **semilog** para distinguir las distintas etapas del entrenamiento.

Se muestran también las contribuciones de la EDO, la condición inicial y las observaciones, sin confundir esas penalizaciones con el error de aproximación.

In [ ]:
# ----------------------------------------------------------
# Función de pérdida de la ecuación logística: escala semilog
# ----------------------------------------------------------
epocas = [fila['epoca'] for fila in historial_logistica]
perdidas = [fila['perdida'] for fila in historial_logistica]

plt.figure(figsize=(8, 4))
plt.semilogy(epocas, perdidas, label='Pérdida total PINN')
plt.xlabel('Épocas')
plt.ylabel('Pérdida total')
plt.title('Ecuación logística: evolución de la pérdida')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
guardar_figura('perdida_logistica.png')

# ----------------------------------------------------------
# Desglose de la función de pérdida del experimento
# ----------------------------------------------------------
# IMPORTANTE: estas curvas son términos de la función objetivo,
# NO los errores relativos L2/H1 respecto de la solución exacta.
# Los valores se muestran en escala semilogarítmica sin aplicar log a los CSV.
import pandas as pd

tabla = pd.DataFrame(historial_logistica)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
# La línea negra suma los términos registrados, con sus pesos efectivos.
ax1.semilogy(tabla['epoca'], tabla['perdida'], color='black', lw=2,
             label='Pérdida total')
ax1.semilogy(tabla['epoca'], tabla['pde'].clip(lower=1e-30), label='Residuo EDP')
ax1.semilogy(tabla['epoca'], tabla['bc'].clip(lower=1e-30), label='Frontera')
ax1.semilogy(tabla['epoca'], tabla['datos'].clip(lower=1e-30), label='Observaciones')
ax1.set(xlabel='Épocas', ylabel='Pérdida (escala logarítmica)',
        title='Evolución de las componentes')
ax1.grid(True, which='both', alpha=.3)
ax1.legend(fontsize=9)

# Participación porcentual: sirve para ver qué término domina el objetivo.
# Todos los términos se evalúan antes de cada actualización de pesos.
componentes = np.column_stack([tabla['pde'].to_numpy(), tabla['bc'].to_numpy(), tabla['datos'].to_numpy()])
totales = componentes.sum(axis=1)
porcentajes = 100 * componentes / np.maximum(totales[:,None], 1e-30)
ax2.stackplot(tabla['epoca'], *porcentajes.T, labels=['Residuo EDP', 'Frontera', 'Observaciones'], alpha=.85)
ax2.set(xlabel='Épocas', ylabel='Contribución a la pérdida (%)',
        title='Participación de cada término', ylim=(0, 100))
ax2.grid(True, alpha=.2)
ax2.legend(loc='upper right', fontsize=8)
plt.tight_layout()
guardar_figura('desglose_perdidas_logistica.png')



# ----------------------------------------------------------
# Figura didáctica: solución (lineal) y pérdidas (semilog)
# ----------------------------------------------------------
# Se replica la disposición del gráfico de referencia de la tesis.
# Las curvas de pérdidas son penalizaciones entrenadas, no error L2/H1.
fig, (ax_sup, ax_inf) = plt.subplots(2, 1, figsize=(10.5, 8.0))
ax_sup.plot(x_eval.detach().cpu().numpy(), f_eval.detach().cpu().numpy(),
            label='Solución neuronal', linewidth=1.8)
ax_sup.plot(x_eval.detach().cpu().numpy(), numeric_solution.y.T,
            '--', label='Solución de referencia', linewidth=1.8)
ax_sup.scatter(x_train.detach().cpu().numpy(), y_train.detach().cpu().numpy(),
               color='tab:blue', label='Observaciones', zorder=4)
ax_sup.set(xlabel='t', ylabel='u(t)', title='Ecuación logística')
ax_sup.grid(alpha=.25)
ax_sup.legend()
tabla_componentes = pd.DataFrame(historial_logistica)
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['perdida'],
                color='black', linewidth=2, label='Pérdida total')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['pde'].clip(lower=1e-30),
                label='Pérdida EDP')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['bc'].clip(lower=1e-30),
                label='Pérdida frontera')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['datos'].clip(lower=1e-30),
                label='Pérdida datos')
ax_inf.set(xlabel='Épocas', ylabel='Contribución a la pérdida',
           title='Evolución de la función de pérdida y sus componentes')
ax_inf.legend()
ax_inf.grid(True, which='both', alpha=.25)
guardar_figura('solucion_y_componentes_logistica.png')


### Ejemplo II: la ecuación de Poisson 1D

- Ahora queremos hallar una función $u(x)$ que satisface la siguiente EDP de segundo orden:

$$\frac{\partial^2 u}{\partial x^2} = -\pi^2 \sin(\pi x)$$

- donde $x \in [-1,1]$.

- Además tenemos las siguientes condiciones de contorno:

$$u(-1)=0$$
$$u(1)=0$$

In [ ]:
# Poisson 1D con datos: seno
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {'ep_poisson1d': 220 if MODO == 'demostracion' else 5000}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Poisson 1D: solución seno con datos:', DEVICE, '| salida:', SALIDA)

import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np

# Definimos el dominio
domain = [-1.0, 1.0]

# Definimos la red neuronal
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.linear_tanh_stack = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        out = self.linear_tanh_stack(x)
        return out

# Definimos la derivada de 2do orden usando PyTorch autograd
def d2f_dx2(f: SimpleNN, x: torch.Tensor) -> torch.Tensor:
    u = f(x)
    du_dx = torch.autograd.grad(
        u, x, grad_outputs=torch.ones_like(u), create_graph=True
    )[0]
    d2u_dx2 = torch.autograd.grad(
        du_dx, x, grad_outputs=torch.ones_like(du_dx), create_graph=True
    )[0]
    return d2u_dx2

# Generamos los puntos de colocación
x_collocation = torch.linspace(domain[0], domain[1], steps=50, requires_grad=True).reshape(-1, 1)

# Entrenamos los data points
x_train = torch.tensor([[-0.8], [-0.4], [0.0], [0.4], [0.8]], dtype=torch.float32)
y_train = torch.sin(torch.pi * x_train)

La función de pérdida está dada por:

$$ \mathcal{L}_\rm{PDE} + \mathcal{L}_\rm{BC} + \mathcal{L}_\rm{data} $$

donde $\mathcal{L}_\rm{PDE}$ está dada por:

$$\mathcal{L}_\rm{PDE}=\frac{1}{n}\sum_{i=1}^{n}\big( \frac{\partial^2 NN}{\partial x^2} \bigg\rvert_{x_i}+ \pi^2 \sin(\pi x_{i})  \big)^2$$

y $\mathcal{L}_\rm{BC}$ está dada por:

$$\mathcal{L}_\rm{BC}=NN(x_{0})^2 + NN(x_{final})^2$$

donde $x_{0}=-1$ y $x_{final}=1$

In [ ]:
# Definimos la función de pérdida
def compute_loss(nn: SimpleNN, x_collocation: torch.Tensor,
                 x_train: torch.Tensor, y_train: torch.Tensor,
                 devolver_componentes: bool = False) -> torch.float:
    # PDE loss
    pde_loss = d2f_dx2(nn, x_collocation) + (torch.pi ** 2) * torch.sin(torch.pi * x_collocation)
    pde_loss = pde_loss.pow(2).mean()

    # Boundary conditions loss
    bc_loss_1 = nn(torch.tensor([domain[0]], requires_grad=True)).pow(2)
    bc_loss_2 = nn(torch.tensor([domain[1]], requires_grad=True)).pow(2)

    # Data loss (MSE)
    mse_loss = torch.nn.MSELoss()(nn(x_train), y_train)

    # Total loss
    tot_loss = pde_loss + bc_loss_1 + bc_loss_2 + mse_loss

    if devolver_componentes:
        return tot_loss, pde_loss, bc_loss_1 + bc_loss_2, mse_loss
    return tot_loss

model = SimpleNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

# Entrenamos el modelo
historial_poisson_seno = []
for ep in range(CFG['ep_poisson1d']):
    loss, j_pde, j_frontera, j_datos = compute_loss(
        model, x_collocation, x_train, y_train, devolver_componentes=True)

    historial_poisson_seno.append({
        'epoca': ep+1, 'perdida': float(loss.detach()),
        'pde': float(j_pde.detach()), 'bc': float(j_frontera.detach()),
        'datos': float(j_datos.detach())})
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    if ep % 500 == 0:
        print(f"Epoch: {ep}, Loss: {loss.item():.6f}")

# Guardar la curva de entrenamiento para su auditoría.
guardar_csv('historial_poisson_seno.csv', historial_poisson_seno)

In [ ]:
# Plotting
x_test = torch.linspace(domain[0], domain[1], steps=100).reshape(-1, 1)
with torch.no_grad():
    y_pred = model(x_test)

# Solución analítica y Solución NN
analytical_sol = torch.sin(torch.pi * x_test)

plt.plot(x_test, y_pred, label="NN Solution")
plt.plot(x_test, analytical_sol, label="Analytical Solution", linestyle='dashed')
plt.scatter(x_train.detach().numpy(), y_train.detach().numpy(), label="Observation Data", color="blue")
plt.xlabel("x")
plt.ylabel("u(x)")
plt.title("Solution to the Poisson Equation with Dirichlet BCs")
plt.legend()
plt.savefig(SALIDA/'poisson_1d_seno_con_datos.png', dpi=180, bbox_inches='tight')
plt.show()

**Evolución de la pérdida.** La disminución de la pérdida durante el entrenamiento describe el proceso de optimización, pero no equivale por sí misma a una cota del error respecto de la solución exacta.

El desglose semilogarítmico permite separar el residuo diferencial, las dos condiciones de frontera y el ajuste a observaciones; las curvas porcentuales indican qué término domina la **pérdida de entrenamiento**.

In [ ]:
# Visualización de la pérdida total (residuo + condiciones de frontera + datos)
plt.figure(figsize=(8, 4))
plt.semilogy([f['epoca'] for f in historial_poisson_seno],
             [f['perdida'] for f in historial_poisson_seno])
plt.xlabel('Época')
plt.ylabel('Pérdida total')
plt.title('Entrenamiento PINN para Poisson 1D: sin(πx)')
plt.grid(True, alpha=0.3)
guardar_figura('perdida_poisson_seno.png')

# ----------------------------------------------------------
# Desglose de la función de pérdida del experimento
# ----------------------------------------------------------
# IMPORTANTE: estas curvas son términos de la función objetivo,
# NO los errores relativos L2/H1 respecto de la solución exacta.
# Los valores se muestran en escala semilogarítmica sin aplicar log a los CSV.
import pandas as pd

tabla = pd.DataFrame(historial_poisson_seno)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
# La línea negra suma los términos registrados, con sus pesos efectivos.
ax1.semilogy(tabla['epoca'], tabla['perdida'], color='black', lw=2,
             label='Pérdida total')
ax1.semilogy(tabla['epoca'], tabla['pde'].clip(lower=1e-30), label='Residuo EDP')
ax1.semilogy(tabla['epoca'], tabla['bc'].clip(lower=1e-30), label='Frontera')
ax1.semilogy(tabla['epoca'], tabla['datos'].clip(lower=1e-30), label='Observaciones')
ax1.set(xlabel='Épocas', ylabel='Pérdida (escala logarítmica)',
        title='Evolución de las componentes')
ax1.grid(True, which='both', alpha=.3)
ax1.legend(fontsize=9)

# Participación porcentual: sirve para ver qué término domina el objetivo.
# Todos los términos se evalúan antes de cada actualización de pesos.
componentes = np.column_stack([tabla['pde'].to_numpy(), tabla['bc'].to_numpy(), tabla['datos'].to_numpy()])
totales = componentes.sum(axis=1)
porcentajes = 100 * componentes / np.maximum(totales[:,None], 1e-30)
ax2.stackplot(tabla['epoca'], *porcentajes.T, labels=['Residuo EDP', 'Frontera', 'Observaciones'], alpha=.85)
ax2.set(xlabel='Épocas', ylabel='Contribución a la pérdida (%)',
        title='Participación de cada término', ylim=(0, 100))
ax2.grid(True, alpha=.2)
ax2.legend(loc='upper right', fontsize=8)
plt.tight_layout()
guardar_figura('desglose_perdidas_poisson_seno.png')



# ----------------------------------------------------------
# Figura didáctica: solución (lineal) y pérdidas (semilog)
# ----------------------------------------------------------
# Se replica la disposición del gráfico de referencia de la tesis.
# Las curvas de pérdidas son penalizaciones entrenadas, no error L2/H1.
fig, (ax_sup, ax_inf) = plt.subplots(2, 1, figsize=(10.5, 8.0))
ax_sup.plot(x_test.detach().cpu().numpy(), y_pred.detach().cpu().numpy(),
            label='Solución PINN', linewidth=1.8)
ax_sup.plot(x_test.detach().cpu().numpy(), analytical_sol.detach().cpu().numpy(),
            '--', label='Solución analítica', linewidth=1.8)
ax_sup.scatter(x_train.detach().cpu().numpy(), y_train.detach().cpu().numpy(),
               color='tab:blue', label='Observaciones', zorder=4)
ax_sup.set(xlabel='x', ylabel='u(x)', title='Poisson 1D con observaciones')
ax_sup.grid(alpha=.25)
ax_sup.legend()
tabla_componentes = pd.DataFrame(historial_poisson_seno)
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['perdida'],
                color='black', linewidth=2, label='Pérdida total')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['pde'].clip(lower=1e-30),
                label='Pérdida EDP')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['bc'].clip(lower=1e-30),
                label='Pérdida frontera')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['datos'].clip(lower=1e-30),
                label='Pérdida datos')
ax_inf.set(xlabel='Épocas', ylabel='Contribución a la pérdida',
           title='Evolución de la función de pérdida y sus componentes')
ax_inf.legend()
ax_inf.grid(True, which='both', alpha=.25)
guardar_figura('solucion_y_componentes_poisson_seno.png')


### Ejemplo III: la ecuación de Poisson 1D (con data points y sin data points)

- Ahora queremos hallar una función $u(x)$ que satisface la siguiente EDP de segundo orden:

$$\frac{\partial^2 u}{\partial x^2} = -2 \operatorname{sech}^2(x)\tanh(x)$$

- donde $x \in ]-1,1[$.

- Además tenemos las siguientes condiciones de contorno:

$$u(-1)=\tanh(-1)$$
$$u(1)=\tanh(1)$$

In [ ]:
# Poisson 1D con datos: tangente hiperbólica
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {'ep_poisson1d': 220 if MODO == 'demostracion' else 5000}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Poisson 1D: tanh con observaciones:', DEVICE, '| salida:', SALIDA)

import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np

# Definimos el dominio
domain = [-1.0, 1.0]

# Definimos la red neuronal
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.linear_tanh_stack = nn.Sequential(
            nn.Linear(1, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 1),
        )

    def forward(self, x):
        out = self.linear_tanh_stack(x)
        return out

# Definimos la derivada de 2do orden usando PyTorch autograd
def d2f_dx2(f: SimpleNN, x: torch.Tensor) -> torch.Tensor:
    u = f(x)
    du_dx = torch.autograd.grad(
        u, x, grad_outputs=torch.ones_like(u), create_graph=True
    )[0]
    d2u_dx2 = torch.autograd.grad(
        du_dx, x, grad_outputs=torch.ones_like(du_dx), create_graph=True
    )[0]
    return d2u_dx2

# Generamos los puntos de colocación
x_collocation = torch.linspace(domain[0], domain[1], steps=50, requires_grad=True).reshape(-1, 1)

# Entrenamos los data points
x_train = torch.tensor([[-0.8], [-0.4], [0.0], [0.4], [0.8]], dtype=torch.float32)
y_train = torch.tanh(x_train)

La función de pérdida está dada por:

$$ \mathcal{L}_\rm{PDE} + \mathcal{L}_\rm{BC} + \mathcal{L}_\rm{data} $$

donde $\mathcal{L}_\rm{PDE}$ está dada por:

$$\mathcal{L}_\rm{PDE}=\frac{1}{n}\sum_{i=1}^{n}\big( \frac{\partial^2 NN}{\partial x^2} \bigg\rvert_{x_i}+ 2\operatorname{sech}^2(x_i)\tanh(x_i)  \big)^2$$

y $\mathcal{L}_\rm{BC}$ está dada por:

$$\mathcal{L}_\rm{BC}=(NN(x_{0})-Tanh(x_{0}))^2 + (NN(x_{final})-Tanh(x_{final}))^2$$

donde $x_{0}=-1$ y $x_{final}=1$

In [ ]:
# Definimos la función de pérdida
def compute_loss(nn: SimpleNN, x_collocation: torch.Tensor,
                 x_train: torch.Tensor, y_train: torch.Tensor,
                 devolver_componentes: bool = False) -> torch.float:
    # PDE loss
    pde_loss = d2f_dx2(nn, x_collocation) + 2 * ((1 / torch.cosh(x_collocation)) ** 2) * torch.tanh(x_collocation)
    pde_loss = pde_loss.pow(2).mean()

    # Boundary conditions loss
    bc_loss_1 = (nn(torch.tensor([domain[0]], requires_grad=True))-torch.tanh(torch.tensor([domain[0]], requires_grad=True))).pow(2)
    bc_loss_2 = (nn(torch.tensor([domain[1]], requires_grad=True))-torch.tanh(torch.tensor([domain[1]], requires_grad=True))).pow(2)

    # Data loss (MSE)
    mse_loss = torch.nn.MSELoss()(nn(x_train), y_train)

    # Total loss
    tot_loss = pde_loss + bc_loss_1 + bc_loss_2 + mse_loss

    if devolver_componentes:
        return tot_loss, pde_loss, bc_loss_1 + bc_loss_2, mse_loss
    return tot_loss

model = SimpleNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

# Histórico de entrenamiento del problema con cinco observaciones.
historial_poisson_tanh_datos = []

# Entrenamos el modelo
for ep in range(CFG['ep_poisson1d']):
    loss, j_pde, j_frontera, j_datos = compute_loss(
        model, x_collocation, x_train, y_train, devolver_componentes=True)

    # Valor de cada componente antes de actualizar los pesos.
    historial_poisson_tanh_datos.append({
        'epoca': ep + 1, 'perdida': float(loss.detach()),
        'pde': float(j_pde.detach()), 'bc': float(j_frontera.detach()),
        'datos': float(j_datos.detach())})

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    if ep % 500 == 0:
        print(f"Epoch: {ep}, Loss: {loss.item():.6f}")
# Permite regenerar la curva de convergencia de la pérdida.
guardar_csv('historial_poisson_tanh_con_datos.csv', historial_poisson_tanh_datos)


In [ ]:
# Plotting
x_test = torch.linspace(domain[0], domain[1], steps=100).reshape(-1, 1)
with torch.no_grad():
    y_pred = model(x_test)

# Solución analítica y Solución NN
analytical_sol = torch.tanh(x_test)

plt.plot(x_test, y_pred, label="NN Solution")
plt.plot(x_test, analytical_sol, label="Analytical Solution", linestyle='dashed')
plt.scatter(x_train.detach().numpy(), y_train.detach().numpy(), label="Observation Data", color="blue")
plt.xlabel("x")
plt.ylabel("u(x)")
plt.title("Solution to the Poisson Equation with Dirichlet BCs")
plt.legend()
plt.savefig(SALIDA/'poisson_1d_tanh_con_datos.png', dpi=180, bbox_inches='tight')
plt.show()

**Evolución de la pérdida con observaciones de $\tanh(x)$.** La solución exacta y la aproximada se comparan en ejes lineales; la pérdida residual, de contorno y de observaciones se representa en **semilog**.

Se conserva por separado la contribución de cada término de la pérdida (EDP, frontera y datos). El error relativo de la solución debe evaluarse de manera independiente.

In [ ]:
# ----------------------------------------------------------
# Evolución de la pérdida para Poisson 1D con observaciones
# ----------------------------------------------------------
epocas = [fila['epoca'] for fila in historial_poisson_tanh_datos]
perdidas = [fila['perdida'] for fila in historial_poisson_tanh_datos]

plt.figure(figsize=(8, 4))
plt.semilogy(epocas, perdidas, label='Pérdida total PINN')
plt.xlabel('Épocas')
plt.ylabel('Pérdida total')
plt.title('Poisson 1D: tanh(x) con observaciones')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
guardar_figura('perdida_poisson_tanh_con_datos.png')

# ----------------------------------------------------------
# Desglose de la función de pérdida del experimento
# ----------------------------------------------------------
# IMPORTANTE: estas curvas son términos de la función objetivo,
# NO los errores relativos L2/H1 respecto de la solución exacta.
# Los valores se muestran en escala semilogarítmica sin aplicar log a los CSV.
import pandas as pd

tabla = pd.DataFrame(historial_poisson_tanh_datos)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
# La línea negra suma los términos registrados, con sus pesos efectivos.
ax1.semilogy(tabla['epoca'], tabla['perdida'], color='black', lw=2,
             label='Pérdida total')
ax1.semilogy(tabla['epoca'], tabla['pde'].clip(lower=1e-30), label='Residuo EDP')
ax1.semilogy(tabla['epoca'], tabla['bc'].clip(lower=1e-30), label='Frontera')
ax1.semilogy(tabla['epoca'], tabla['datos'].clip(lower=1e-30), label='Observaciones')
ax1.set(xlabel='Épocas', ylabel='Pérdida (escala logarítmica)',
        title='Evolución de las componentes')
ax1.grid(True, which='both', alpha=.3)
ax1.legend(fontsize=9)

# Participación porcentual: sirve para ver qué término domina el objetivo.
# Todos los términos se evalúan antes de cada actualización de pesos.
componentes = np.column_stack([tabla['pde'].to_numpy(), tabla['bc'].to_numpy(), tabla['datos'].to_numpy()])
totales = componentes.sum(axis=1)
porcentajes = 100 * componentes / np.maximum(totales[:,None], 1e-30)
ax2.stackplot(tabla['epoca'], *porcentajes.T, labels=['Residuo EDP', 'Frontera', 'Observaciones'], alpha=.85)
ax2.set(xlabel='Épocas', ylabel='Contribución a la pérdida (%)',
        title='Participación de cada término', ylim=(0, 100))
ax2.grid(True, alpha=.2)
ax2.legend(loc='upper right', fontsize=8)
plt.tight_layout()
guardar_figura('desglose_perdidas_poisson_tanh_con_datos.png')



# ----------------------------------------------------------
# Figura didáctica: solución (lineal) y pérdidas (semilog)
# ----------------------------------------------------------
# Se replica la disposición del gráfico de referencia de la tesis.
# Las curvas de pérdidas son penalizaciones entrenadas, no error L2/H1.
fig, (ax_sup, ax_inf) = plt.subplots(2, 1, figsize=(10.5, 8.0))
ax_sup.plot(x_test.detach().cpu().numpy(), y_pred.detach().cpu().numpy(),
            label='Solución PINN', linewidth=1.8)
ax_sup.plot(x_test.detach().cpu().numpy(), analytical_sol.detach().cpu().numpy(),
            '--', label='Solución analítica', linewidth=1.8)
ax_sup.scatter(x_train.detach().cpu().numpy(), y_train.detach().cpu().numpy(),
               color='tab:blue', label='Observaciones', zorder=4)
ax_sup.set(xlabel='x', ylabel='u(x)', title='Poisson 1D con observaciones')
ax_sup.grid(alpha=.25)
ax_sup.legend()
tabla_componentes = pd.DataFrame(historial_poisson_tanh_datos)
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['perdida'],
                color='black', linewidth=2, label='Pérdida total')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['pde'].clip(lower=1e-30),
                label='Pérdida EDP')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['bc'].clip(lower=1e-30),
                label='Pérdida frontera')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['datos'].clip(lower=1e-30),
                label='Pérdida datos')
ax_inf.set(xlabel='Épocas', ylabel='Contribución a la pérdida',
           title='Evolución de la función de pérdida y sus componentes')
ax_inf.legend()
ax_inf.grid(True, which='both', alpha=.25)
guardar_figura('solucion_y_componentes_poisson_tanh_con_datos.png')


### Poisson 1D sin observaciones interiores

En esta configuración se eliminan los puntos de observación de la solución. La pérdida solo penaliza el residuo diferencial y las condiciones de Dirichlet. Se compara con la misma solución exacta $u^*(x)=\tanh(x)$ y se miden los errores relativos en $L^2$ y $H^1$.

In [ ]:
# Poisson 1D sin observaciones interiores
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {'ep_poisson1d': 220 if MODO == 'demostracion' else 5000}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Poisson 1D: tanh sin observaciones:', DEVICE, '| salida:', SALIDA)

import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np

# --- Configuración ---
domain = [-1.0, 1.0]
EPOCHS = CFG['ep_poisson1d']
N_COLLOCATION = 100

# --- Red neuronal ---
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 1)
        )

    def forward(self, x):
        return self.net(x)

# --- Derivada segunda ---
def d2f_dx2(f, x):
    x.requires_grad_(True)
    u = f(x)
    du = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    d2u = torch.autograd.grad(du, x, grad_outputs=torch.ones_like(du), create_graph=True)[0]
    return d2u

# --- Función de pérdida ---
def compute_loss(model, x_collocation, devolver_componentes=False):
    # PDE residual
    d2u = d2f_dx2(model, x_collocation)
    f_rhs = -2 * (1 / torch.cosh(x_collocation))**2 * torch.tanh(x_collocation)
    pde_loss = ((d2u - f_rhs) ** 2).mean()

    # Condiciones de frontera
    x_bc = torch.tensor([[-1.0], [1.0]], requires_grad=True)
    u_bc = torch.tanh(x_bc)
    bc_loss = torch.nn.MSELoss()(model(x_bc), u_bc)

    perdida_total = pde_loss + bc_loss
    if devolver_componentes:
        return perdida_total, pde_loss, bc_loss
    return perdida_total

# --- Entrenamiento ---
model = PINN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)

x_collocation = torch.linspace(domain[0], domain[1], N_COLLOCATION).reshape(-1, 1)
x_collocation.requires_grad_(True)

# Registramos la pérdida para analizar el entrenamiento.
historial_poisson_tanh_sin_datos = []

for epoch in range(EPOCHS):
    loss, j_pde, j_frontera = compute_loss(
        model, x_collocation, devolver_componentes=True)
    historial_poisson_tanh_sin_datos.append({
        'epoca': epoch + 1, 'perdida': float(loss.detach()),
        'pde': float(j_pde.detach()), 'bc': float(j_frontera.detach())})
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4e}")

# --- Comparación visual ---
x_plot = torch.linspace(domain[0], domain[1], 500).reshape(-1, 1)
u_pred = model(x_plot).detach()
u_exact = torch.tanh(x_plot)

plt.figure(figsize=(10, 6))
plt.plot(x_plot, u_exact, label='Solución exacta', linewidth=2)
plt.plot(x_plot, u_pred, '--', label='Solución PINN', linewidth=2)
plt.xlabel('x')
plt.ylabel('u(x)')
plt.title('Solución de la EDP de Poisson 1D con PINN')
plt.legend()
plt.grid(True)
# La solución se representa linealmente, sin transformar u(x).
guardar_figura('poisson_1d_tanh_sin_datos.png')

# --- Cálculo de errores L2 y H1 ---
def compute_l2_and_h1_errors(model, x_grid):
    x_grid = x_grid.clone().detach().requires_grad_(True)
    u_pred = model(x_grid)
    u_true = torch.tanh(x_grid)

    du_pred = torch.autograd.grad(u_pred, x_grid, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    du_true = 1 - torch.tanh(x_grid)**2  # sech²(x)

    dx = x_grid[1] - x_grid[0]

    l2_num = torch.sqrt(torch.sum((u_pred - u_true) ** 2) * dx)
    l2_den = torch.sqrt(torch.sum((u_true) ** 2) * dx)
    l2_rel_error = (l2_num / l2_den).item()

    h1_num = torch.sqrt(torch.sum((du_pred - du_true) ** 2) * dx)
    h1_den = torch.sqrt(torch.sum((du_true) ** 2) * dx)
    h1_rel_error = torch.sqrt((l2_num**2 + h1_num**2) / (l2_den**2 + h1_den**2)).item()

    return l2_rel_error, h1_rel_error

x_fine = torch.linspace(domain[0], domain[1], 1000).reshape(-1, 1)
l2_err, h1_err = compute_l2_and_h1_errors(model, x_fine)

print(f"\n✅ Error relativo L2: {l2_err:.4e}")
print(f"✅ Error relativo H1: {h1_err:.4e}")


# Guardar las métricas del experimento sin datos interiores.
guardar_csv('errores_poisson_1d_sin_datos.csv', [{'error_L2_rel': l2_err, 'error_H1_rel': h1_err, 'epocas': EPOCHS, 'semilla': SEMILLA}])
# Guardamos la historia original de pérdidas junto con los errores.
guardar_csv('historial_poisson_tanh_sin_datos.csv', historial_poisson_tanh_sin_datos)


**Evolución de la pérdida sin observaciones interiores.** Los puntos de colocación y las condiciones de Dirichlet siguen presentes. Como antes, la solución se representa en escala lineal y la pérdida en escala **semilogarítmica**.

La comparación de componentes incluye exclusivamente residuo de Poisson y Dirichlet, puesto que no se dispone de observaciones interiores en este experimento.

In [ ]:
# ----------------------------------------------------------
# Evolución de la pérdida para Poisson 1D sin observaciones
# ----------------------------------------------------------
epocas = [fila['epoca'] for fila in historial_poisson_tanh_sin_datos]
perdidas = [fila['perdida'] for fila in historial_poisson_tanh_sin_datos]

plt.figure(figsize=(8, 4))
plt.semilogy(epocas, perdidas, label='Pérdida total PINN')
plt.xlabel('Épocas')
plt.ylabel('Pérdida total')
plt.title('Poisson 1D: tanh(x) sin observaciones')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
guardar_figura('perdida_poisson_tanh_sin_datos.png')

# ----------------------------------------------------------
# Desglose de la función de pérdida del experimento
# ----------------------------------------------------------
# IMPORTANTE: estas curvas son términos de la función objetivo,
# NO los errores relativos L2/H1 respecto de la solución exacta.
# Los valores se muestran en escala semilogarítmica sin aplicar log a los CSV.
import pandas as pd

tabla = pd.DataFrame(historial_poisson_tanh_sin_datos)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
# La línea negra suma los términos registrados, con sus pesos efectivos.
ax1.semilogy(tabla['epoca'], tabla['perdida'], color='black', lw=2,
             label='Pérdida total')
ax1.semilogy(tabla['epoca'], tabla['pde'].clip(lower=1e-30), label='Residuo EDP')
ax1.semilogy(tabla['epoca'], tabla['bc'].clip(lower=1e-30), label='Frontera')
ax1.set(xlabel='Épocas', ylabel='Pérdida (escala logarítmica)',
        title='Evolución de las componentes')
ax1.grid(True, which='both', alpha=.3)
ax1.legend(fontsize=9)

# Participación porcentual: sirve para ver qué término domina el objetivo.
# Todos los términos se evalúan antes de cada actualización de pesos.
componentes = np.column_stack([tabla['pde'].to_numpy(), tabla['bc'].to_numpy()])
totales = componentes.sum(axis=1)
porcentajes = 100 * componentes / np.maximum(totales[:,None], 1e-30)
ax2.stackplot(tabla['epoca'], *porcentajes.T, labels=['Residuo EDP', 'Frontera'], alpha=.85)
ax2.set(xlabel='Épocas', ylabel='Contribución a la pérdida (%)',
        title='Participación de cada término', ylim=(0, 100))
ax2.grid(True, alpha=.2)
ax2.legend(loc='upper right', fontsize=8)
plt.tight_layout()
guardar_figura('desglose_perdidas_poisson_tanh_sin_datos.png')



# ----------------------------------------------------------
# Figura didáctica: solución (lineal) y pérdidas (semilog)
# ----------------------------------------------------------
# Se replica la disposición del gráfico de referencia de la tesis.
# Las curvas de pérdidas son penalizaciones entrenadas, no error L2/H1.
fig, (ax_sup, ax_inf) = plt.subplots(2, 1, figsize=(10.5, 8.0))
ax_sup.plot(x_plot.detach().cpu().numpy(), u_pred.detach().cpu().numpy(),
            label='Solución PINN', linewidth=1.8)
ax_sup.plot(x_plot.detach().cpu().numpy(), u_exact.detach().cpu().numpy(),
            '--', label='Solución analítica', linewidth=1.8)
ax_sup.set(xlabel='x', ylabel='u(x)', title='Poisson 1D sin observaciones interiores')
ax_sup.grid(alpha=.25)
ax_sup.legend()
tabla_componentes = pd.DataFrame(historial_poisson_tanh_sin_datos)
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['perdida'],
                color='black', linewidth=2, label='Pérdida total')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['pde'].clip(lower=1e-30),
                label='Pérdida EDP')
ax_inf.semilogy(tabla_componentes['epoca'], tabla_componentes['bc'].clip(lower=1e-30),
                label='Pérdida frontera')
ax_inf.set(xlabel='Épocas', ylabel='Contribución a la pérdida',
           title='Evolución de la función de pérdida y sus componentes')
ax_inf.legend()
ax_inf.grid(True, which='both', alpha=.25)
guardar_figura('solucion_y_componentes_poisson_tanh_sin_datos.png')


### Comparación PINN estándar, regularización cuadrática media y LIPR

Para el problema de Poisson 1D

$$-u''(x)=2\tanh(x)\operatorname{sech}^{2}(x),\qquad x\in(-1,1),\qquad u^*(x)=\tanh(x),$$

se comparan **tres funciones de pérdida distintas**. Si $r_\theta(x)=-u''_\theta(x)-f(x)$,

$$
\begin{aligned}
\mathcal J_{\mathrm{PINN}}(\theta)&=\frac{1}{N_f}\sum_i |r_\theta(x_i)|^2 + \mathcal J_{\mathrm{BC}},\\
\mathcal J_{\mathrm{media}}(\theta)&=\mathcal J_{\mathrm{PINN}}(\theta)+\lambda_R\frac{1}{N_f}\sum_i |u_\theta'''(x_i)|^2,\\
\mathcal J_{\mathrm{LIPR}}(\theta)&=\mathcal J_{\mathrm{PINN}}(\theta)+\lambda_R\max_i|u_\theta'''(x_i)|^2.
\end{aligned}
$$

La tercera función es **la regularización Lipschitz discreta de Shin, Darbon y Karniadakis (2020), ecuación (4.1)**, aplicada al operador $\mathcal L[u]=-u''$. En efecto, $\partial_x\mathcal L[u_\theta]=-u_\theta'''$, por lo que $\max_i|\partial_x\mathcal L[u_\theta](x_i)|^2=\max_i|u_\theta'''(x_i)|^2$.

La penalización de frontera es $\mathcal J_{\mathrm{BC}}=\tfrac12\bigl(|u_\theta(-1)-u^*(-1)|^2+|u_\theta(1)-u^*(1)|^2\bigr)$: corresponde a $\lambda_b=1$ en ese experimento. Se toma $\lambda_R=N_f^{-3/2}$ y se emplean idénticos puntos e inicialización para los tres modelos.

**Distinción teórica importante.** La ecuación (3.3) del artículo utiliza una *seminorma de Hölder continua*. La LIPR de la ecuación (3.4), concretada en (4.1), **la aproxima computacionalmente mediante máximos en una muestra**, pero no es igual a dicha seminorma. El experimento 2D de este cuaderno utiliza la **pérdida PINN estándar**, sin regularizadores LIPR de interior ni de frontera. Además, aunque el funcional LIPR 1D coincide, aquí se utiliza un MLP con dos capas ocultas de 50 neuronas en lugar de la arquitectura residual particular de la sección 4.1 de Shin; por eso se reproduce **la formulación de la pérdida**, no todos los detalles de su experimento.

**Fuente:** Shin, Darbon y Karniadakis, *Communications in Computational Physics*, 28(5), 2042–2074 (2020), ecuaciones (3.3), (3.4) y (4.1), DOI: [10.4208/cicp.OA-2020-0193](https://doi.org/10.4208/cicp.OA-2020-0193).


In [ ]:
# PINN y regularización de tercer orden
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {
    'n_reg': [6,16] if MODO=='demostracion' else [10,20,40,80,160,320,640],
    'semillas': [SEMILLA] if MODO=='demostracion' else [SEMILLA,SEMILLA+1,SEMILLA+2],
    'adam_reg': 35 if MODO=='demostracion' else 4000,
    'lbfgs_reg': 0 if MODO=='demostracion' else 350,
}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('PINN / regularización media / LIPR:', DEVICE, '| salida:', SALIDA)

# ----------------------------------------------------------
# Redes y derivadas para el experimento de regularización
# ----------------------------------------------------------
from torch.autograd import grad

class RedPoissonRegularizada(nn.Module):
    """MLP completamente conectada con dos capas ocultas de 50 neuronas."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 50), nn.Tanh(),
            nn.Linear(50, 50), nn.Tanh(),
            nn.Linear(50, 1)
        )

    def forward(self, x):
        return self.net(x)

def derivadas_hasta_tercera(modelo, x):
    # x debe permitir diferenciación. Devuelve u, u_x, u_xx y u_xxx.
    u = modelo(x)
    ux = grad(u, x, torch.ones_like(u), create_graph=True)[0]
    uxx = grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    uxxx = grad(uxx, x, torch.ones_like(uxx), create_graph=True)[0]
    return u, ux, uxx, uxxx

def fuente_poisson_tanh(x):
    # Si u(x) = tanh(x), entonces -u''(x) = 2*tanh(x)*sech^2(x).
    return 2.0 * torch.tanh(x) / torch.cosh(x).square()

def componentes_perdida(modelo, x_f, x_b, metodo, lambda_r):
    """Calcula la pérdida PINN, media cuadrática o LIPR de Shin (4.1).

    En PINN estándar basta u_xx. Las derivadas u_xxx se calculan
    únicamente para las dos variantes regularizadas, evitando trabajo extra.
    """
    x = x_f.detach().requires_grad_(True)
    u = modelo(x)
    ux = grad(u, x, torch.ones_like(u), create_graph=True)[0]
    uxx = grad(ux, x, torch.ones_like(ux), create_graph=True)[0]

    # 1) Residuo de la EDP: -u''(x) = 2*tanh(x)*sech^2(x).
    residuo = -uxx - fuente_poisson_tanh(x)
    perdida_pde = residuo.square().mean()

    # 2) Dirichlet en los dos extremos; media = 1/2 de la suma de cuadrados.
    perdida_bc = (modelo(x_b) - torch.tanh(x_b)).square().mean()
    perdida_base = perdida_pde + perdida_bc

    # 3) Regularización. El máximo de Shin NO es una media.
    if metodo == 'PINN':
        regularizador = x.new_zeros(())
    elif metodo in ('Media', 'LIPR-Shin'):
        uxxx = grad(uxx, x, torch.ones_like(uxx), create_graph=True)[0]
        cuadrados = uxxx.square()
        if metodo == 'Media':
            regularizador = cuadrados.mean()
        else:
            # Shin (4.1): max_i |d L[u]/dx (x_i)|^2,
            # para L[u]=-u_xx equivale a max_i |u_xxx(x_i)|^2.
            # Selección explícita del índice máximo: conserva la derivada
            # respecto de todos los parámetros, incluidos los sesgos.
            posicion = cuadrados.detach().reshape(-1).argmax()
            regularizador = cuadrados.reshape(-1)[posicion]
    else:
        raise ValueError(f'Método desconocido: {metodo}')

    perdida_total = perdida_base + lambda_r * regularizador
    return perdida_total, perdida_pde, perdida_bc, regularizador

def errores_relativos_1d(modelo, n_eval=1001):
    """Error L2 y error H1, integrados en una cuadrícula independiente."""
    x = torch.linspace(-1, 1, n_eval, device=DEVICE,
                       dtype=DTYPE_REG).reshape(-1, 1).requires_grad_(True)
    u = modelo(x)
    ux = grad(u, x, torch.ones_like(u), create_graph=False)[0]
    exacta = torch.tanh(x)
    exacta_x = 1.0 - exacta.square()

    # Regla trapezoidal. Incluye u y su derivada en la norma H1.
    e2 = torch.trapz((u - exacta).square().flatten(), x.flatten())
    n2 = torch.trapz(exacta.square().flatten(), x.flatten())
    e_der = torch.trapz((ux - exacta_x).square().flatten(), x.flatten())
    n_der = torch.trapz(exacta_x.square().flatten(), x.flatten())
    return (float(torch.sqrt(e2/n2).detach().cpu()),
            float(torch.sqrt((e2+e_der)/(n2+n_der)).detach().cpu()))

In [ ]:
# ----------------------------------------------------------
# Comprobación de la identidad usada en LIPR de Shin (4.1)
# ----------------------------------------------------------
# Se trata de una verificación del funcional discreto, no del
# teorema de convergencia ni de la seminorma continua de Hölder.
fijar_semilla(SEMILLA)
modelo_comprobacion = RedPoissonRegularizada().to(DEVICE, dtype=DTYPE_REG)
xi = torch.linspace(-1, 1, 9, device=DEVICE,
                    dtype=DTYPE_REG).reshape(-1, 1)
xb = torch.tensor([[-1.0], [1.0]], device=DEVICE, dtype=DTYPE_REG)
lambda_prueba = 9 ** (-1.5)

# Calcular directamente max |u_xxx|² a partir de autodiferenciación.
_x = xi.clone().detach().requires_grad_(True)
_, _, _uxx, _uxxx = derivadas_hasta_tercera(modelo_comprobacion, _x)
maximo_manual = _uxxx.square().max()
media_manual = _uxxx.square().mean()

perdida_lipr, _, _, regularizador_lipr = componentes_perdida(
    modelo_comprobacion, xi, xb, 'LIPR-Shin', lambda_prueba)
perdida_media, _, _, regularizador_media = componentes_perdida(
    modelo_comprobacion, xi, xb, 'Media', lambda_prueba)
perdida_pinn, _, _, _ = componentes_perdida(
    modelo_comprobacion, xi, xb, 'PINN', lambda_prueba)

assert torch.allclose(regularizador_lipr, maximo_manual, atol=1e-11, rtol=1e-9)
assert torch.allclose(regularizador_media, media_manual, atol=1e-11, rtol=1e-9)
assert regularizador_lipr >= regularizador_media
assert torch.allclose(perdida_lipr-perdida_pinn,
                      lambda_prueba*maximo_manual, atol=1e-11, rtol=1e-9)
assert torch.allclose(perdida_media-perdida_pinn,
                      lambda_prueba*media_manual, atol=1e-11, rtol=1e-9)
print('Verificación de la pérdida LIPR discreta (Shin, Ec. 4.1): correcta.')
print(f'Regularizador máximo: {regularizador_lipr.item():.6e}')
print(f'Regularizador medio:  {regularizador_media.item():.6e}')


In [ ]:
# ----------------------------------------------------------
# Entrenamiento comparable de los tres métodos
# ----------------------------------------------------------
import copy
METODOS = ['PINN', 'Media', 'LIPR-Shin']
registros_regularizacion = []
# Guardamos solo un caso representativo para mostrar la pérdida por época.
# Todos los tamaños y semillas siguen formando parte del barrido de errores.
historial_regularizacion_adam = []

for N_f in CFG['n_reg']:
    # Puntos equiespaciados del problema [-1,1].
    puntos_f = torch.linspace(-1, 1, N_f, device=DEVICE,
                              dtype=DTYPE_REG).reshape(-1, 1)
    puntos_b = torch.tensor([[-1.0], [1.0]], device=DEVICE, dtype=DTYPE_REG)
    lambda_r = N_f ** (-1.5)

    for semilla in CFG['semillas']:
        fijar_semilla(semilla)
        red_inicial = RedPoissonRegularizada().to(DEVICE, dtype=DTYPE_REG)
        estado_inicial = copy.deepcopy(red_inicial.state_dict())

        for metodo in METODOS:
            # MISMA inicialización y MISMOS puntos para comparar los métodos.
            modelo = RedPoissonRegularizada().to(DEVICE, dtype=DTYPE_REG)
            modelo.load_state_dict(estado_inicial)
            optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
            comienzo = time.perf_counter()

            # Fase I: optimización por Adam.
            for epoca in range(CFG['adam_reg']):
                optimizador.zero_grad(set_to_none=True)
                perdida, j_pde, j_frontera, r_sin_peso = componentes_perdida(
                    modelo, puntos_f, puntos_b, metodo, lambda_r
                )
                if not bool(torch.isfinite(perdida).item()):
                    raise FloatingPointError(f'Pérdida no finita en {N_f}/{metodo}')
                # Registrar el proceso Adam para el mayor N_f y la primera semilla.
                # La pérdida de cada método es diferente: sus valores absolutos
                # NO son una comparación directa de precisión entre modelos.
                if (N_f == max(CFG['n_reg']) and semilla == CFG['semillas'][0]
                    and (epoca == 0 or (epoca + 1) % max(1, CFG['adam_reg']//100) == 0
                         or epoca + 1 == CFG['adam_reg'])):
                    historial_regularizacion_adam.append({
                        'epoca': epoca + 1, 'N_f': N_f, 'semilla': semilla,
                        'metodo': metodo, 'perdida_total': float(perdida.detach()),
                        'pde': float(j_pde.detach()),
                        'bc': float(j_frontera.detach()),
                        'regularizacion_ponderada': float((lambda_r*r_sin_peso).detach())
                    })
                perdida.backward()
                optimizador.step()

            # Fase II: refinamiento con L-BFGS (si se ha solicitado).
            if CFG['lbfgs_reg'] > 0:
                optim_lbfgs = torch.optim.LBFGS(
                    modelo.parameters(), lr=1.0,
                    max_iter=CFG['lbfgs_reg'], history_size=20,
                    line_search_fn='strong_wolfe'
                )
                def closure():
                    optim_lbfgs.zero_grad(set_to_none=True)
                    perdida, _, _, _ = componentes_perdida(
                        modelo, puntos_f, puntos_b, metodo, lambda_r
                    )
                    perdida.backward()
                    return perdida
                optim_lbfgs.step(closure)

            segundos = time.perf_counter() - comienzo
            error_l2, error_h1 = errores_relativos_1d(modelo)
            fila = {
                'N_f': N_f, 'semilla': semilla, 'metodo': metodo,
                'lambda_R': lambda_r,
                'error_L2_rel': error_l2, 'error_H1_rel': error_h1,
                'segundos': segundos,
                'adam_epocas': CFG['adam_reg'],
                'lbfgs_max_iter': CFG['lbfgs_reg'],
                'dispositivo': str(DEVICE), 'protocolo': MODO,
            }
            registros_regularizacion.append(fila)
            # Guardado progresivo: no perder los casos ya ejecutados.
            guardar_csv('regularizacion_casos.csv', registros_regularizacion)
            print(f'N={N_f:4d} | {metodo:10s} | semilla={semilla} | '
                  f'L2={error_l2:.3e} | H1={error_h1:.3e} | {segundos:.1f} s')
# Historial SOLO de Adam; si se activa L-BFGS, su fase no se mezcla en este eje.
guardar_csv('historial_regularizacion_adam.csv', historial_regularizacion_adam)


In [ ]:
# ----------------------------------------------------------
# Curvas de convergencia de los errores relativos L2 y H1
# ----------------------------------------------------------
df_reg = pd.DataFrame(registros_regularizacion)
resumen = (df_reg.groupby(['N_f', 'metodo'], as_index=False)
           [['error_L2_rel', 'error_H1_rel', 'segundos']].median())
resumen.to_csv(SALIDA/'regularizacion_resumen.csv', index=False)

plt.figure(figsize=(10, 7))
colores = {'PINN': 'tab:blue', 'Media': 'tab:green', 'LIPR-Shin': 'tab:red'}
for metodo in METODOS:
    tabla = resumen[resumen['metodo'] == metodo].sort_values('N_f')
    plt.loglog(tabla['N_f'], tabla['error_L2_rel'], 'o-',
               label=f'{metodo} — L2', color=colores[metodo])
    plt.loglog(tabla['N_f'], tabla['error_H1_rel'], 's--',
               label=f'{metodo} — H1', color=colores[metodo])
plt.xlabel('Número de puntos de colocación $N_f$')
plt.ylabel('Error relativo')
plt.title('Poisson 1D: PINN estándar y regularización de orden superior')
plt.grid(True, which='both', alpha=0.35)
plt.legend(ncol=2)
guardar_figura('figura_4_7_regularizacion.png')

# Las medianas son resultados experimentales del optimizador; no tasas demostradas.

**Pérdida durante la optimización de la comparación de regularizadores.** La Figura 4.7 muestra los errores relativos $L^2$ y $H^1$ en **escala log-log**, adecuados para estudiar su evolución al aumentar $N_f$. Como complemento, se representa en **semilog** la pérdida durante Adam para un tamaño y semilla representativos. Como los tres métodos minimizan funciones objetivo diferentes, una pérdida total menor no significa necesariamente un error de aproximación menor. Si se utiliza L-BFGS, este gráfico describe solo la fase Adam.

Las curvas adicionales muestran los términos del funcional de cada método. Para LIPR y regularización media se grafica $\lambda_R\,\mathcal R$ (regularización **ponderada**), que es su contribución efectiva a la pérdida optimizada.

In [ ]:
# ----------------------------------------------------------
# Pérdidas de entrenamiento: PINN, media cuadrática y LIPR
# ----------------------------------------------------------
tabla_historial = pd.DataFrame(historial_regularizacion_adam)
plt.figure(figsize=(9, 5))

for metodo in METODOS:
    datos_metodo = tabla_historial[tabla_historial['metodo'] == metodo]
    datos_metodo = datos_metodo.sort_values('epoca')
    plt.semilogy(datos_metodo['epoca'], datos_metodo['perdida_total'],
                 label=metodo)

plt.xlabel('Épocas de optimización Adam')
plt.ylabel('Función de pérdida total de cada método')
plt.title(f'Pérdidas de entrenamiento (N_f={max(CFG["n_reg"])})')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
guardar_figura('perdidas_regularizacion_adam.png')

# ----------------------------------------------------------
# Desglose de las tres funciones de pérdida del estudio LIPR
# ----------------------------------------------------------
# No confundimos la magnitud del regularizador con el error L2/H1.
# La contribución regularizada es λ_R × R, no R sin ponderar.
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)
for eje, metodo in zip(ejes, METODOS):
    datos = tabla_historial[tabla_historial['metodo'] == metodo].sort_values('epoca')
    eje.semilogy(datos['epoca'], datos['perdida_total'], color='black',
                 linewidth=2, label='Total')
    eje.semilogy(datos['epoca'], datos['pde'].clip(lower=1e-30), label='EDP')
    eje.semilogy(datos['epoca'], datos['bc'].clip(lower=1e-30), label='Frontera')
    if metodo != 'PINN':
        eje.semilogy(datos['epoca'],
                     datos['regularizacion_ponderada'].clip(lower=1e-30),
                     label=r'$\lambda_R\,R$')
    eje.set_title(metodo)
    eje.set_xlabel('Épocas Adam')
    eje.grid(True, which='both', alpha=.25)
    eje.legend(fontsize=8)
ejes[0].set_ylabel('Componentes de la pérdida (semilog)')
fig.suptitle(f'Contribuciones al objetivo (N_f={max(CFG["n_reg"])})', fontsize=12)
plt.tight_layout()
guardar_figura('desglose_perdidas_regularizacion.png')


## Reproducción de los experimentos elípticos de Shin, Darbon y Karniadakis

Se implementan los dos problemas elípticos presentados en la sección 4.1 de los dos experimentos de la sección 4.1 del artículo [Shin, Darbon y Karniadakis (2020)](https://doi.org/10.4208/cicp.OA-2020-0193): **Figura 3**, $u^*(x)=\tanh(x)$ con frontera de Dirichlet impuesta débilmente, y **Figura 4**, $u^*(x)=(1-x^2)\sin(6\pi x)$ con frontera impuesta exactamente.

La arquitectura es la red **residual** de dos capas ocultas y 50 neuronas:

$$N_\theta(x)=W^3\tanh\!\left(W^2\left(\mathbf 1x+\tanh(W^1x+b^1)\right)+b^2\right)+b^3.$$

Se utiliza inicialización Xavier, Adam seguido de L-BFGS, muestreo equiespaciado, pérdida PINN y la LIPR **discreta** de la ecuación (4.1):

$$J_{\mathrm{LIPR}}(\theta)=J_{\mathrm{PINN}}(\theta)+m_r^{-3/2}\max_{x_i\in T_r}|h_\theta^{(3)}(x_i)|^2.$$

En la **Figura 3**, las condiciones correctas son $u(\pm1)=\tanh(\pm1)$ (aunque la ecuación (4.1) del preprint escriba una frontera homogénea). En la **Figura 4** la función que entra al operador es $h_\theta(x)=(1-x^2)N_\theta(x)$, por lo que el borde es exactamente cero.

**Parámetros de ejecución:** `SHIN_PROTOCOLO='exploracion'` genera las mismas clases de figuras mediante una ejecución acotada y reproducible; `SHIN_PROTOCOLO='articulo'` utiliza tamaños de muestra que cubren los rangos del artículo y las épocas declaradas por los autores. Los puntos precisos de la malla de tamaños, el número de iteraciones L-BFGS y las elecciones de implementación que el artículo no especifica se registran en archivos CSV/JSON; los resultados no deben etiquetarse como reproducción exacta si se usa el protocolo acotado.

**Error graficado:** discreto **no normalizado** sobre una cuadrícula de prueba fija:

$$E_{L^2,K}=\sqrt{K^{-1}\sum_k|h_\theta(z_k)-u^*(z_k)|^2},\quad
E_{H^1,K}=\sqrt{E_{L^2,K}^2+K^{-1}\sum_k|h'_\theta(z_k)-(u^*)'(z_k)|^2}.$$

Estas cantidades son diferentes de los errores **relativos** de la sección anterior.

**Ejecución autónoma:** iniciar este apartado en la celda de la red residual; contiene importaciones, constantes, semillas y checkpoints propios.


In [ ]:
# Problemas elípticos de Shin
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Reproducciones de Shin, Figuras 3 y 4:', DEVICE, '| salida:', SALIDA)

# Protocolo de reproducción publicado frente a exploración reducida.
SHIN_PROTOCOLO = 'articulo'  # 'exploracion' o 'articulo'
SHIN_GUARDAR_CADA_ADAM = 250
SHIN_GUARDAR_CADA_LBFGS = 25
SHIN_REANUDAR = True

# ----------------------------------------------------------
# Red residual y funciones exactas de los ejemplos de Shin
# ----------------------------------------------------------
from torch.autograd import grad
import copy

class RedResidualShin(nn.Module):
    """Red residual de dos capas ocultas y ancho 50, según la sección 4.1."""
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(1, 50)
        self.capa2 = nn.Linear(50, 50)
        self.salida = nn.Linear(50, 1)
        for modulo in [self.capa1, self.capa2, self.salida]:
            nn.init.xavier_uniform_(modulo.weight)
            nn.init.zeros_(modulo.bias)

    def forward(self, x):
        z1 = torch.tanh(self.capa1(x))
        # 1*x replica la entrada escalar en las 50 neuronas.
        z2 = x.expand(-1, 50) + z1
        return self.salida(torch.tanh(self.capa2(z2)))

class RedShinDirichletFuerte(nn.Module):
    """Cumple h(-1)=h(1)=0 exactamente para todo theta."""
    def __init__(self):
        super().__init__()
        self.base = RedResidualShin()

    def forward(self, x):
        return (1.0 - x.square()) * self.base(x)


def exacta_shin(x, caso):
    if caso == 'tanh':
        return torch.tanh(x)
    if caso == 'oscilatorio':
        return (1.0-x.square())*torch.sin(6.0*torch.pi*x)
    raise ValueError(caso)


def derivada_exacta_shin(x, caso):
    if caso == 'tanh':
        return 1.0-torch.tanh(x).square()
    if caso == 'oscilatorio':
        k = 6.0*torch.pi
        return -2.0*x*torch.sin(k*x) + k*(1.0-x.square())*torch.cos(k*x)
    raise ValueError(caso)


def fuente_shin(x, caso):
    # Definimos L[h]=-h'' y por tanto f=-u*''.
    if caso == 'tanh':
        return 2*torch.tanh(x)/torch.cosh(x).square()
    k = 6.0*torch.pi
    return (2+k*k*(1-x.square()))*torch.sin(k*x) + 4*k*x*torch.cos(k*x)


def perdida_shin(modelo, x_interior, caso, metodo):
    """Devuelve J_TOTAL, J_PDE, J_BC y R_LIPR."""
    x = x_interior.detach().clone().requires_grad_(True)
    pred = modelo(x)
    ux = grad(pred, x, torch.ones_like(pred), create_graph=True)[0]
    uxx = grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    residuo = -uxx-fuente_shin(x,caso)
    jpde = residuo.square().mean()

    if caso == 'tanh':
        frontera = torch.tensor([[-1.0],[1.0]],device=DEVICE,dtype=DTYPE_REG)
        jbc = (modelo(frontera)-torch.tanh(frontera)).square().mean()
    else:
        # Dirichlet fuerte: el contorno está satisfecho exactamente.
        jbc = jpde.new_zeros(())

    regularizador = jpde.new_zeros(())
    if metodo == 'LIPR':
        uxxx = grad(uxx, x, torch.ones_like(uxx), create_graph=True)[0]
        cuadrados = uxxx.square().reshape(-1)
        indice = cuadrados.detach().argmax()
        regularizador = cuadrados[indice]
    elif metodo != 'PINN':
        raise ValueError(metodo)
    lambda_r = len(x_interior)**(-1.5)
    return jpde + jbc + lambda_r*regularizador, jpde, jbc, regularizador


def errores_shin_discretos(modelo, caso, puntos=2000, bloque=400):
    """Errores absolutos discretos L2 y H1; evaluación independiente por bloques."""
    eu, ed, em = [], [], []
    for inicio in range(0,puntos,bloque):
        fin = min(inicio+bloque,puntos)
        x = torch.linspace(-1,1,puntos,device=DEVICE,dtype=DTYPE_REG)[inicio:fin].reshape(-1,1)
        x = x.detach().requires_grad_(True)
        u = modelo(x)
        ux = grad(u,x,torch.ones_like(u),create_graph=False)[0]
        eu.extend((u-exacta_shin(x,caso)).detach().cpu().flatten().numpy())
        ed.extend((ux-derivada_exacta_shin(x,caso)).detach().cpu().flatten().numpy())
    eu,ed=np.asarray(eu),np.asarray(ed)
    return float(np.sqrt(np.mean(eu**2))), float(np.sqrt(np.mean(eu**2+ed**2))), float(np.max(np.abs(eu)))

# Verificación de la fuente: u*'' + f = 0 en ambos ejemplos.
for _caso in ('tanh','oscilatorio'):
    _x=torch.linspace(-0.98,0.98,19,device=DEVICE,dtype=DTYPE_REG).reshape(-1,1).requires_grad_(True)
    _u=exacta_shin(_x,_caso)
    _ux=grad(_u,_x,torch.ones_like(_u),create_graph=True)[0]
    _uxx=grad(_ux,_x,torch.ones_like(_ux),create_graph=False)[0]
    _desfase=float((_uxx+fuente_shin(_x,_caso)).abs().max().detach().cpu())
    assert _desfase<1e-8,(_caso,_desfase)
print('Funciones fuente y soluciones exactas: comprobadas.')

### Parámetros de las dos reproducciones y ejecución en GPU

La **Figura 3** usa frontera débil, lote completo, $\lambda_b=1$ y hasta 25 000 épocas de Adam en el protocolo del artículo. La **Figura 4** usa frontera fuerte, minilotes de 100 y entre 5 000 y 25 000 épocas, según el ensayo. En ambos casos, el artículo indica Adam seguido de L-BFGS, pero no especifica todas las tolerancias ni todas las semillas.

La configuración de exploración permite examinar el comportamiento del código con menos épocas; para contrastar las curvas de manera más próxima a las condiciones publicadas con las Figuras 3 y 4 del artículo se debe ejecutar `SHIN_PROTOCOLO='articulo'` y conservar los metadatos. Esta modalidad puede requerir una GPU de gran memoria, sesiones prolongadas y/o dividir los tamaños de muestra entre sesiones. **Los errores no se precalculan**.

Para tamaños grandes se acumulan gradientes por bloques, manteniendo el funcional de lote completo. Esta estrategia reduce memoria de GPU, **pero no reduce el elevado costo de entrenamiento del protocolo del artículo**.

**Almacenamiento de entrenamientos prolongados.** Los entrenamientos largos guardan checkpoints periódicos en
`checkpoints_shin/`, con modelo, optimizador, época o iteración y estados aleatorios. El formato
puede recuperarse únicamente desde archivos de confianza. Para que sobrevivan a un reinicio
del entorno hay que activar `GUARDAR_EN_DRIVE=True`. La segmentación de L-BFGS en grupos
de iteraciones conserva su historial de curvatura, pero puede producir un recorrido numérico
ligeramente distinto al de una única llamada monolítica.


In [ ]:
# ----------------------------------------------------------
# Protocolo computacional reproducible (Shin, figuras 3 y 4)
# ----------------------------------------------------------
if SHIN_PROTOCOLO == 'articulo':
    SHIN_CFG = {
        'tanh': {'n':[1,2,4,8,10,20,40,80,160,320,640,1280,2500,5000,10000],
                 'adam':25000,'lbfgs':250,'eval':10000,'batch':None},
        'oscilatorio': {'n':[50,100,200,500,1000,2000,3500,5000],
                       'adam':5000,'lbfgs':250,'eval':10000,'batch':100},
    }
else:
    SHIN_CFG = {
        'tanh': {'n':[1,4,10,40,160,640],
                 'adam':(25 if MODO=='demostracion' else 750),
                 'lbfgs':(0 if MODO=='demostracion' else 65),
                 'eval':(301 if MODO=='demostracion' else 2001),'batch':None},
        'oscilatorio': {'n':[50,100,250,500],
                       'adam':(25 if MODO=='demostracion' else 900),
                       'lbfgs':(0 if MODO=='demostracion' else 65),
                       'eval':(301 if MODO=='demostracion' else 2001),'batch':100},
    }
# Los valores concretos se guardan para poder repetir resultados.
(SALIDA/'protocolo_shin.json').write_text(
    json.dumps({'modo_shin':SHIN_PROTOCOLO,'opciones':SHIN_CFG,
                'xavier':'uniforme','adam_lr':1e-3,
                'lbfgs_lr':1.0,'lbfgs_tol':None,
                'semilla':SEMILLA,'dtype':'float64',
                'checkpoint_adam_cada':SHIN_GUARDAR_CADA_ADAM,
                'checkpoint_lbfgs_cada':SHIN_GUARDAR_CADA_LBFGS,
                'nota':'El artículo no publica todas las tolerancias/semillas ni cada tamaño de muestra.'},
               indent=2,ensure_ascii=False),encoding='utf8')

# Las ejecuciones largas pueden reanudarse si se guardan en Google Drive.
def rejilla_shin(n):
    if n==1:
        return torch.zeros((1,1),device=DEVICE,dtype=DTYPE_REG)
    return torch.linspace(-1,1,n,device=DEVICE,dtype=DTYPE_REG).reshape(-1,1)

def perdida_shin_por_bloques_y_gradiente(modelo, x_full, caso, metodo, bloque=256):
    """Computa el objetivo FULL-BATCH y acumula gradientes sin grafo gigante.

    Para LIPR busca el máximo en todos los puntos y calcula el gradiente
    únicamente en el punto que alcanza ese máximo. Es el subgradiente
    del máximo discreto cuando el punto maximizador es único.
    Debe llamarse DESPUÉS de optim.zero_grad().
    """
    n_total=len(x_full)
    pde_total=0.0
    indice_maximo=None
    valor_maximo=-1.0
    for ini in range(0,n_total,bloque):
        x=x_full[ini:ini+bloque].detach().clone().requires_grad_(True)
        u=modelo(x)
        ux=grad(u,x,torch.ones_like(u),create_graph=True)[0]
        uxx=grad(ux,x,torch.ones_like(ux),create_graph=True)[0]
        residuo=-uxx-fuente_shin(x,caso)
        loss=(residuo.square().sum()/n_total)
        pde_total+=float(loss.detach().cpu())
        loss.backward()
        if metodo=='LIPR':
            # Nueva evaluación para buscar el máximo sin conservar
            # los grafos de todos los bloques simultáneamente.
            z=x_full[ini:ini+bloque].detach().clone().requires_grad_(True)
            uz=modelo(z)
            du=grad(uz,z,torch.ones_like(uz),create_graph=True)[0]
            d2u=grad(du,z,torch.ones_like(du),create_graph=True)[0]
            d3u=grad(d2u,z,torch.ones_like(d2u),create_graph=False)[0]
            zvalores=d3u.detach().square().flatten()
            valor_local,ind_local=zvalores.max(dim=0)
            if float(valor_local)>valor_maximo:
                valor_maximo=float(valor_local)
                indice_maximo=ini+int(ind_local)

    total=pde_total
    if caso=='tanh':
        borde=torch.tensor([[-1.0],[1.0]],device=DEVICE,dtype=DTYPE_REG)
        condicion=(modelo(borde)-torch.tanh(borde)).square().mean()
        total+=float(condicion.detach().cpu())
        condicion.backward()

    if metodo=='LIPR':
        punto=x_full[indice_maximo:indice_maximo+1].detach().clone().requires_grad_(True)
        uv=modelo(punto)
        dv=grad(uv,punto,torch.ones_like(uv),create_graph=True)[0]
        d2v=grad(dv,punto,torch.ones_like(dv),create_graph=True)[0]
        d3v=grad(d2v,punto,torch.ones_like(d2v),create_graph=True)[0]
        regularizacion=n_total**(-1.5)*d3v.square().sum()
        total+=float(regularizacion.detach().cpu())
        regularizacion.backward()
    return torch.tensor(total,dtype=DTYPE_REG,device=DEVICE)


# ----------------------------------------------------------
# Guardado y recuperación de entrenamientos largos de Shin
# ----------------------------------------------------------
# Se guarda después de completar una época de Adam o un bloque de L-BFGS.
# Al reiniciar Colab, los archivos .pt se leen desde el mismo directorio
# persistente del que se recuperan los CSV ya finalizados.
import shutil

CARPETA_CHECKPOINTS_SHIN = SALIDA / 'checkpoints_shin'
CARPETA_CHECKPOINTS_SHIN.mkdir(parents=True, exist_ok=True)


def ruta_checkpoint_shin(caso, n, metodo, cfg):
    identificador = f'{SHIN_PROTOCOLO}_{caso}_{metodo}_N{n}_A{cfg["adam"]}'
    return CARPETA_CHECKPOINTS_SHIN / f'{identificador}.pt'


def contexto_checkpoint_shin(caso, n, metodo, cfg):
    # Impide restaurar por accidente otro experimento o protocolo.
    return {
        'caso': caso, 'N_f': int(n), 'metodo': metodo,
        'semilla': int(SEMILLA), 'protocolo': SHIN_PROTOCOLO,
        'cfg': dict(cfg), 'dtype': str(DTYPE_REG),
        'modelo': ('RedResidualShin' if caso == 'tanh'
                   else 'RedShinDirichletFuerte'),
        'lbfgs_bloque': int(SHIN_GUARDAR_CADA_LBFGS),
    }


def estados_aleatorios_shin():
    # Guardar también el muestreo aleatorio de minilotes del ejemplo oscilatorio.
    return {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
        'torch_cuda': torch.cuda.get_rng_state_all()
                      if torch.cuda.is_available() else None,
    }


def recuperar_aleatoriedad_shin(estados):
    random.setstate(estados['python'])
    np.random.set_state(estados['numpy'])
    torch.set_rng_state(estados['torch_cpu'].cpu())
    if estados.get('torch_cuda') is not None and torch.cuda.is_available():
        if len(estados['torch_cuda']) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(estados['torch_cuda'])
        else:
            print('Aviso: cantidad de GPU diferente; no se garantiza el mismo muestreo.')


def guardar_checkpoint_shin(ruta, contenido):
    """Guardado por archivo temporal y copia de respaldo, útil en Drive montado."""
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_suffix('.temporal.pt')
    respaldo = ruta.with_suffix('.anterior.pt')
    # No sobrescribir la última copia válida sin conservar una alternativa.
    torch.save(contenido, temporal)
    if ruta.exists():
        shutil.copy2(ruta, respaldo)
    os.replace(temporal, ruta)


def leer_checkpoint_shin(ruta, contexto):
    if not SHIN_REANUDAR:
        return None
    # El respaldo se utiliza si el guardado más reciente quedó incompleto.
    for candidato in [ruta, ruta.with_suffix('.anterior.pt')]:
        if not candidato.exists():
            continue
        try:
            # weights_only=False: incluye estados de NumPy/Python.
            # Abrir ÚNICAMENTE checkpoints generados por este notebook.
            estado = torch.load(candidato, map_location='cpu', weights_only=False)
        except Exception as exc:
            print('Checkpoint ilegible:', candidato.name, repr(exc))
            continue
        if estado.get('version') != 1 or estado.get('contexto') != contexto:
            print('Checkpoint incompatible, no se utilizará:', candidato.name)
            continue
        if candidato != ruta:
            print('Recuperado desde la copia de respaldo:', candidato.name)
        return estado
    return None


def eliminar_checkpoint_shin(caso, n, metodo, cfg):
    """Se ejecuta solo DESPUÉS de que el resultado final figure en el CSV."""
    ruta = ruta_checkpoint_shin(caso, n, metodo, cfg)
    for objetivo in [ruta, ruta.with_suffix('.anterior.pt'),
                     ruta.with_suffix('.temporal.pt')]:
        if objetivo.exists():
            objetivo.unlink()


def entrenar_shin(caso, n, metodo, cfg, estado):
    # 1. Preparar red y puntos según el protocolo original.
    modelo = (RedResidualShin() if caso == 'tanh'
              else RedShinDirichletFuerte()).to(DEVICE, dtype=DTYPE_REG)
    modelo.load_state_dict(estado)
    x_full = rejilla_shin(n)
    optim_adam = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    ruta = ruta_checkpoint_shin(caso, n, metodo, cfg)
    contexto = contexto_checkpoint_shin(caso, n, metodo, cfg)

    # 2. Recuperar un entrenamiento incompleto, si existe.
    anterior = leer_checkpoint_shin(ruta, contexto)
    epoca_inicial = 0
    etapa = 'adam'
    historia = []
    segundos_anteriores = 0.0
    estado_lbfgs = None
    iteraciones_lbfgs = 0

    if anterior is not None:
        modelo.load_state_dict(anterior['modelo'])
        etapa = anterior['etapa']
        epoca_inicial = int(anterior['epocas_adam'])
        iteraciones_lbfgs = int(anterior['iteraciones_lbfgs'])
        historia = list(anterior['historia'])
        segundos_anteriores = float(anterior['segundos_acumulados'])
        estado_lbfgs = anterior.get('optim_lbfgs')
        if anterior.get('optim_adam') is not None:
            optim_adam.load_state_dict(anterior['optim_adam'])
        recuperar_aleatoriedad_shin(anterior['rng'])
        print(f'REANUDADO {caso} N={n} {metodo}: '
              f'Adam={epoca_inicial}/{cfg["adam"]}, '
              f'L-BFGS={iteraciones_lbfgs}/{cfg["lbfgs"]}, etapa={etapa}')

        # Si terminó el entrenamiento, pero Colab cayó antes de guardar el CSV:
        if etapa == 'completado':
            return modelo, anterior['fila_final'], historia

    inicio_tramo = time.perf_counter()

    def checkpoint(etapa_actual, epocas, lbfgs_realizadas, optim_lbfgs=None,
                   fila_final=None):
        # Guardar un estado consistente al FINAL de un paso de optimización.
        total_segundos = segundos_anteriores + (time.perf_counter() - inicio_tramo)
        paquete = {
            'version': 1, 'contexto': contexto,
            'etapa': etapa_actual,
            'epocas_adam': int(epocas),
            'iteraciones_lbfgs': int(lbfgs_realizadas),
            'modelo': modelo.state_dict(),
            'optim_adam': optim_adam.state_dict(),
            'optim_lbfgs': (optim_lbfgs.state_dict()
                            if optim_lbfgs is not None else estado_lbfgs),
            'historia': historia, 'segundos_acumulados': total_segundos,
            'rng': estados_aleatorios_shin(),
            'fila_final': fila_final,
        }
        guardar_checkpoint_shin(ruta, paquete)

    # 3. Entrenamiento Adam (reanuda desde la última época completada).
    if etapa == 'adam':
        for ep in range(epoca_inicial, cfg['adam']):
            # En el caso oscilatorio se usan minilotes de 100 puntos.
            if cfg['batch'] is not None and len(x_full) > cfg['batch']:
                indices = torch.randperm(len(x_full), device=DEVICE)[:cfg['batch']]
                x = x_full[indices]
            else:
                x = x_full

            optim_adam.zero_grad(set_to_none=True)
            if len(x) == len(x_full) and len(x_full) > 1024:
                loss = perdida_shin_por_bloques_y_gradiente(
                    modelo, x_full, caso, metodo)
            else:
                loss, lpde, lbc, reg = perdida_shin(modelo, x, caso, metodo)
                if not torch.isfinite(loss):
                    raise FloatingPointError((caso, metodo, n, ep))
                loss.backward()
            if not torch.isfinite(loss):
                raise FloatingPointError((caso, metodo, n, ep))
            optim_adam.step()

            epocas_hechas = ep + 1
            if (ep == 0 or epocas_hechas == cfg['adam'] or
                    epocas_hechas % max(1, cfg['adam']//100) == 0):
                historia.append({'epoca': epocas_hechas,
                                'perdida': float(loss.detach().cpu()),
                                'metodo': metodo, 'N_f': n, 'caso': caso})

            if (epocas_hechas % SHIN_GUARDAR_CADA_ADAM == 0 or
                    epocas_hechas == cfg['adam']):
                checkpoint('adam', epocas_hechas, 0)
                print(f'  Checkpoint Adam: {caso} N={n} {metodo} '
                      f'({epocas_hechas}/{cfg["adam"]})')

        epoca_inicial = cfg['adam']
        etapa = 'lbfgs'

    # 4. L-BFGS con estado de curvatura persistente. Los bloques permiten
    #    guardar un checkpoint ENTRE llamadas al optimizador.
    if cfg['lbfgs']:
        optim = torch.optim.LBFGS(
            modelo.parameters(), lr=1.0,
            max_iter=max(1, min(SHIN_GUARDAR_CADA_LBFGS, cfg['lbfgs'])),
            history_size=20, line_search_fn='strong_wolfe')
        if estado_lbfgs is not None:
            optim.load_state_dict(estado_lbfgs)

        def cierre():
            optim.zero_grad(set_to_none=True)
            if len(x_full) > 1024:
                return perdida_shin_por_bloques_y_gradiente(
                    modelo, x_full, caso, metodo)
            loss, _, _, _ = perdida_shin(modelo, x_full, caso, metodo)
            loss.backward()
            return loss

        while iteraciones_lbfgs < cfg['lbfgs']:
            pendientes = cfg['lbfgs'] - iteraciones_lbfgs
            optim.param_groups[0]['max_iter'] = min(
                SHIN_GUARDAR_CADA_LBFGS, pendientes)
            optim.step(cierre)
            # PyTorch conserva n_iter dentro del estado del primer parámetro.
            estado_interno = optim.state[optim._params[0]]
            nuevo_total = int(estado_interno.get('n_iter', 0))
            if nuevo_total <= iteraciones_lbfgs:
                print('L-BFGS sin nuevos pasos: detenido por tolerancia.')
                break
            iteraciones_lbfgs = nuevo_total
            checkpoint('lbfgs', cfg['adam'], iteraciones_lbfgs, optim)
            print(f'  Checkpoint L-BFGS: {caso} N={n} {metodo} '
                  f'({iteraciones_lbfgs}/{cfg["lbfgs"]})')

        # Si L-BFGS se detuvo por convergencia, conservar el estado final.
        checkpoint('lbfgs', cfg['adam'], iteraciones_lbfgs, optim)

    # 5. Evaluación independiente y checkpoint de finalización antes del CSV.
    segundos = segundos_anteriores + (time.perf_counter() - inicio_tramo)
    l2, h1, linf = errores_shin_discretos(modelo, caso, puntos=cfg['eval'])
    fila = {
        'caso': caso, 'metodo': metodo, 'N_f': n, 'semilla': SEMILLA,
        'error_L2_discreto': l2, 'error_H1_discreto': h1,
        'error_C0_evaluado': linf, 'segundos': segundos,
        'adam_epocas': cfg['adam'], 'lbfgs_max_iter': cfg['lbfgs'],
        'lbfgs_iteraciones_reales': iteraciones_lbfgs,
        'K_eval': cfg['eval'], 'protocolo_shin': SHIN_PROTOCOLO,
        'checkpoint_adam_cada': SHIN_GUARDAR_CADA_ADAM,
        'checkpoint_lbfgs_cada': SHIN_GUARDAR_CADA_LBFGS,
    }
    checkpoint('completado', cfg['adam'], iteraciones_lbfgs, fila_final=fila)
    return modelo, fila, historia

# Comprobación adicional: la frontera fuerte es exactamente nula.
_xb=torch.tensor([[-1.0],[1.0]],dtype=DTYPE_REG,device=DEVICE)
assert RedShinDirichletFuerte().to(DEVICE,dtype=DTYPE_REG)(_xb).abs().max().item()==0.0
print('Configuración y contorno fuerte: comprobados.')

In [ ]:
# ----------------------------------------------------------
# Reproducción computacional: Figura 3 y Figura 4 de Shin
# ----------------------------------------------------------
# El emparejamiento de PINN y LIPR se hace con la misma inicialización.
from pathlib import Path

for caso in ['tanh','oscilatorio']:
    cfg=SHIN_CFG[caso]
    nombre_csv=f'resultados_shin_{caso}.csv'
    anterior=SALIDA/nombre_csv
    if anterior.exists():
        existentes=pd.read_csv(anterior).to_dict('records')
        # Evitar mezclar protocolos y tamaños tras modificar la configuración.
        existentes=[r for r in existentes if (r.get('protocolo_shin')==SHIN_PROTOCOLO
                    and int(r.get('adam_epocas',-1))==cfg['adam']
                    and int(r.get('K_eval',-1))==cfg['eval'])]
    else:
        existentes=[]
    realizados={(int(f['N_f']),f['metodo']) for f in existentes}
    resultados_shin=list(existentes)
    for n in cfg['n']:
        # La misma inicialización para los dos métodos en un N dado.
        fijar_semilla(SEMILLA+n+(0 if caso=='tanh' else 50000))
        nuevo=(RedResidualShin() if caso=='tanh' else RedShinDirichletFuerte()).to(DEVICE,dtype=DTYPE_REG)
        base=copy.deepcopy(nuevo.state_dict())
        for metodo in ['PINN','LIPR']:
            if (n,metodo) in realizados:
                print('Se conserva el caso ya ejecutado:',caso,n,metodo)
                continue
            fijar_semilla(SEMILLA+n+(0 if caso=='tanh' else 50000))
            _,fila,historia=entrenar_shin(caso,n,metodo,cfg,base)
            resultados_shin.append(fila)
            # Registrar primero el historial: el CSV global se escribe al final.
            guardar_csv(f'historial_shin_{caso}_{metodo}_N{n}.csv',historia)
            guardar_csv(nombre_csv,resultados_shin)
            # Solo retirar el checkpoint tras conservar ambos CSV.
            eliminar_checkpoint_shin(caso,n,metodo,cfg)
            print(f'{caso:11s} N={n:5d} {metodo:5s}: '
                  f'L2={fila["error_L2_discreto"]:.3e}, '
                  f'H1={fila["error_H1_discreto"]:.3e}, '
                  f'{fila["segundos"]:.1f}s')
    print('Total',caso,len(resultados_shin),'filas.')

In [ ]:
# ----------------------------------------------------------
# Figuras comparables con la Figura 3 y la Figura 4 de Shin
# ----------------------------------------------------------
# Ambas usan errores discretos no normalizados y ejes log-log.
for caso,nombre in [('tanh','reproduccion_shin_figura3.png'),
                    ('oscilatorio','reproduccion_shin_figura4.png')]:
    tabla=pd.read_csv(SALIDA/f'resultados_shin_{caso}.csv')
    plt.figure(figsize=(9.2,6))
    for metodo,color,estilo in [('PINN','tab:blue','-.'),('LIPR','tab:red','-')]:
        filas=tabla[tabla.metodo==metodo].sort_values('N_f')
        plt.loglog(filas.N_f,filas.error_L2_discreto,estilo+'x',color=color,
                   label=r'$L^2$ '+metodo,linewidth=1.8,markersize=7)
        plt.loglog(filas.N_f,filas.error_H1_discreto,estilo+'o',color=color,
                   label=r'$H^1$ '+metodo,linewidth=1.8,markersize=5)
    # La línea de referencia tiene pendiente -1, no se ajusta a resultados.
    eje=np.array([min(SHIN_CFG[caso]['n']),max(SHIN_CFG[caso]['n'])],dtype=float)
    positivos=tabla[['error_L2_discreto','error_H1_discreto']].to_numpy().ravel()
    positivos=positivos[np.isfinite(positivos)&(positivos>0)]
    constante=(float(np.median(positivos))*np.sqrt(eje.prod())) if len(positivos) else 1.0
    plt.loglog(eje,constante/eje,':',color='black',label=r'Referencia $O(N_f^{-1})$')
    plt.xlabel(r'Número de puntos residuales $N_f$')
    plt.ylabel(r'Error discreto absoluto (no normalizado)')
    cond=r'Dirichlet débil; $u^*=\tanh(x)$' if caso=='tanh' else r'Dirichlet fuerte; $u^*=(1-x^2)\sin(6\pi x)$'
    plt.title('Poisson 1D: '+cond)
    plt.grid(True,which='both',alpha=.3)
    plt.legend(ncol=2,fontsize=9)
    plt.suptitle(f'Protocolo de Shin: {SHIN_PROTOCOLO}',y=1.03,fontsize=10)
    guardar_figura(nombre)
print('Figuras de Shin creadas en:',SALIDA)

## Regularización Hölder y cobertura: interpretación computacional del capítulo III

La **pérdida Hölder continua** estudiada por Shin, Darbon y Karniadakis [ecuación (3.3)] tiene, para un problema elíptico en una dimensión, la forma

$$
\mathcal J_{H,\alpha}(\theta)=\mathcal J_{\mathrm{PINN}}(\theta)
+\lambda_{r,m}^{R}\,[\mathcal L[h_\theta]]_{\alpha;I}^{2},
\qquad
[v]_{\alpha;I}=\sup_{x\neq y\in I}
\frac{|v(x)-v(y)|}{|x-y|^{\alpha}},\quad 0<\alpha\leq 1.
$$

Se regulariza el **operador** $\mathcal L[h_\theta]=-h_\theta''$, no solamente el residuo $\mathcal L[h_\theta]-f$. Para $d=1$, Shin no añade una penalización Hölder de frontera a (3.3). Para $d\geq2$, el funcional teórico **sí añade** $\lambda_{b,m}^{R}[\mathcal B[h_\theta]]_{\alpha;\Gamma}^{2}$. El experimento numérico de este apartado es exclusivamente **unidimensional**.

**Escala del peso teórico.** El Teorema 3.1, ecuación (3.1), junto a la condición (3.2), proporciona para $d=1$ una escala $\lambda_{r,m}^{R}\asymp m^{-\alpha-1/2}$, con constantes dependientes de la hipótesis de muestreo y de otros parámetros. En el diagnóstico fijamos un prefactor ilustrativo $c_R=1$, por lo que $\lambda_{r,m}^{R}=c_Rm^{-\alpha-1/2}$; esto **no reproduce las constantes específicas** de la cota teórica. La elección $m^{-3/2}$ corresponde a $\alpha=1$ y se utiliza también en la experiencia LIPR de Shin.

El supremo continuo de (3.3) **no puede evaluarse exactamente con una malla finita**. Por ello, el entrenamiento utiliza la *aproximación por pares* en una malla auxiliar $Q_K$ separada de las muestras residuales $T_r$:

$$S_{K,\alpha}(h_\theta)=
\max_{\substack{i<j\\z_i,z_j\in Q_K}}
\frac{|\mathcal L[h_\theta](z_i)-\mathcal L[h_\theta](z_j)|}{|z_i-z_j|^\alpha}
\leq[\mathcal L[h_\theta]]_{\alpha;I}.$$

Se distingue así entre **Hölder continuo (teoría)**, **Hölder aproximado (pares de puntos auxiliares)** y **LIPR (máximo de la derivada en puntos de entrenamiento)**. Para $\alpha=1$ y suficiente suavidad, $[\mathcal L[h]]_{1;I}=\sup_I |h^{(3)}|$; comparar las estimaciones por pares con máximos de $|h^{(3)}|$ en mallas cada vez más densas permite estudiar la relación con LIPR. Con $\alpha=1/2$ se examina una seminorma verdaderamente fraccionaria, ejecutando de nuevo este apartado con ese exponente.

Las figuras muestran picos residuales, refinamiento de la estimación Hölder, pérdidas, error uniforme de prueba, errores relativos y radio de cobertura. La verificación de una desigualdad en **una malla finita** no certifica la desigualdad continua ni demuestra las hipótesis del teorema.




In [ ]:
# Regularidad Hölder y cobertura
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Diagnóstico Hölder y cobertura:', DEVICE, '| salida:', SALIDA)

# Para conectar directamente (3.3) con LIPR de Shin se usa alpha=1.
# Cambiar a 0.5 y ejecutar esta sección otra vez para estudiar Holder fraccionario.
ALPHA_HOLDER = 1.0
FACTOR_PESO_HOLDER = 1.0  # prefactor ilustrativo, NO constante certificada (3.1)

# ----------------------------------------------------------
# Redes y derivadas para el experimento de regularización
# ----------------------------------------------------------
from torch.autograd import grad

class RedPoissonRegularizada(nn.Module):
    """MLP completamente conectada con dos capas ocultas de 50 neuronas."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 50), nn.Tanh(),
            nn.Linear(50, 50), nn.Tanh(),
            nn.Linear(50, 1)
        )

    def forward(self, x):
        return self.net(x)

def derivadas_hasta_tercera(modelo, x):
    # x debe permitir diferenciación. Devuelve u, u_x, u_xx y u_xxx.
    u = modelo(x)
    ux = grad(u, x, torch.ones_like(u), create_graph=True)[0]
    uxx = grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    uxxx = grad(uxx, x, torch.ones_like(uxx), create_graph=True)[0]
    return u, ux, uxx, uxxx

def fuente_poisson_tanh(x):
    # Si u(x) = tanh(x), entonces -u''(x) = 2*tanh(x)*sech^2(x).
    return 2.0 * torch.tanh(x) / torch.cosh(x).square()

def componentes_perdida(modelo, x_f, x_b, metodo, lambda_r):
    """Calcula la pérdida PINN, media cuadrática o LIPR de Shin (4.1).

    En PINN estándar basta u_xx. Las derivadas u_xxx se calculan
    únicamente para las dos variantes regularizadas, evitando trabajo extra.
    """
    x = x_f.detach().requires_grad_(True)
    u = modelo(x)
    ux = grad(u, x, torch.ones_like(u), create_graph=True)[0]
    uxx = grad(ux, x, torch.ones_like(ux), create_graph=True)[0]

    # 1) Residuo de la EDP: -u''(x) = 2*tanh(x)*sech^2(x).
    residuo = -uxx - fuente_poisson_tanh(x)
    perdida_pde = residuo.square().mean()

    # 2) Dirichlet en los dos extremos; media = 1/2 de la suma de cuadrados.
    perdida_bc = (modelo(x_b) - torch.tanh(x_b)).square().mean()
    perdida_base = perdida_pde + perdida_bc

    # 3) Regularización. El máximo de Shin NO es una media.
    if metodo == 'PINN':
        regularizador = x.new_zeros(())
    elif metodo in ('Media', 'LIPR-Shin'):
        uxxx = grad(uxx, x, torch.ones_like(uxx), create_graph=True)[0]
        cuadrados = uxxx.square()
        if metodo == 'Media':
            regularizador = cuadrados.mean()
        else:
            # Shin (4.1): max_i |d L[u]/dx (x_i)|^2,
            # para L[u]=-u_xx equivale a max_i |u_xxx(x_i)|^2.
            # Selección explícita del índice máximo: conserva la derivada
            # respecto de todos los parámetros, incluidos los sesgos.
            posicion = cuadrados.detach().reshape(-1).argmax()
            regularizador = cuadrados.reshape(-1)[posicion]
    else:
        raise ValueError(f'Método desconocido: {metodo}')

    perdida_total = perdida_base + lambda_r * regularizador
    return perdida_total, perdida_pde, perdida_bc, regularizador

def errores_relativos_1d(modelo, n_eval=1001):
    """Error L2 y error H1, integrados en una cuadrícula independiente."""
    x = torch.linspace(-1, 1, n_eval, device=DEVICE,
                       dtype=DTYPE_REG).reshape(-1, 1).requires_grad_(True)
    u = modelo(x)
    ux = grad(u, x, torch.ones_like(u), create_graph=False)[0]
    exacta = torch.tanh(x)
    exacta_x = 1.0 - exacta.square()

    # Regla trapezoidal. Incluye u y su derivada en la norma H1.
    e2 = torch.trapz((u - exacta).square().flatten(), x.flatten())
    n2 = torch.trapz(exacta.square().flatten(), x.flatten())
    e_der = torch.trapz((ux - exacta_x).square().flatten(), x.flatten())
    n_der = torch.trapz(exacta_x.square().flatten(), x.flatten())
    return (float(torch.sqrt(e2/n2).detach().cpu()),
            float(torch.sqrt((e2+e_der)/(n2+n_der)).detach().cpu()))

def rejilla_shin(n):
    # Cuadrícula auxiliar, utilizada solo en el modo determinista.
    if n == 1:
        return torch.zeros((1,1), device=DEVICE, dtype=DTYPE_REG)
    return torch.linspace(-1, 1, n, device=DEVICE, dtype=DTYPE_REG).reshape(-1,1)

# ----------------------------------------------------------
# Cocientes de Hölder y mediciones sobre una malla independiente
# ----------------------------------------------------------
def aproximar_holder(puntos, valores, alpha=0.5, bloque=256):
    """Máximo de cocientes Hölder sobre TODOS los pares i<j, por bloques."""
    x=puntos.detach().reshape(-1).cpu().numpy().astype(float)
    v=valores.detach().reshape(-1).cpu().numpy().astype(float)
    assert len(x)==len(np.unique(x)), 'Los puntos auxiliares deben ser distintos.'
    mejor=0.0
    for i in range(0,len(x),bloque):
        xx=x[i:i+bloque,None]
        vv=v[i:i+bloque,None]
        diferencias=np.abs(xx-x[None,:])
        cocientes=np.divide(np.abs(vv-v[None,:]),
            np.maximum(diferencias,1e-300)**alpha)
        # La diagonal tiene cociente 0: dos puntos iguales se excluyen.
        cocientes[diferencias==0]=0.0
        mejor=max(mejor,float(cocientes.max()))
    return mejor


def evaluar_operador(modelo, x_np, bloque=256):
    """Valores de -h''(x) sin retener el grafo del entrenamiento."""
    salidas=[]
    for inicio in range(0,len(x_np),bloque):
        x=torch.as_tensor(x_np[inicio:inicio+bloque],device=DEVICE,
              dtype=DTYPE_REG).reshape(-1,1).detach().requires_grad_(True)
        u=modelo(x)
        ux=grad(u,x,torch.ones_like(u),create_graph=True)[0]
        uxx=grad(ux,x,torch.ones_like(ux),create_graph=False)[0]
        salidas.append((-uxx).detach().cpu().flatten().numpy())
    return np.concatenate(salidas)



def evaluar_gradiente_operador_1d(modelo, x_np, bloque=256):
    """|d(L[h])/dx|=|h^(3)| en puntos de prueba, no supremo continuo."""
    salidas=[]
    for inicio in range(0,len(x_np),bloque):
        x=torch.as_tensor(x_np[inicio:inicio+bloque],device=DEVICE,
              dtype=DTYPE_REG).reshape(-1,1).detach().requires_grad_(True)
        u=modelo(x)
        ux=grad(u,x,torch.ones_like(u),create_graph=True)[0]
        uxx=grad(ux,x,torch.ones_like(ux),create_graph=True)[0]
        uxxx=grad(uxx,x,torch.ones_like(uxx),create_graph=False)[0]
        salidas.append(uxxx.abs().detach().cpu().reshape(-1).numpy())
    return np.concatenate(salidas)


def perdida_holder_discreta(modelo,x_f,x_b,x_aux,alpha,peso):
    # Penaliza diferencias del OPERADOR -u'', no su tercera derivada.
    x=x_f.detach().clone().requires_grad_(True)
    u=modelo(x); ux=grad(u,x,torch.ones_like(u),create_graph=True)[0]
    uxx=grad(ux,x,torch.ones_like(ux),create_graph=True)[0]
    jpde=(-uxx-fuente_poisson_tanh(x)).square().mean()
    jbc=(modelo(x_b)-torch.tanh(x_b)).square().mean()
    z=x_aux.detach().clone().requires_grad_(True)
    uz=modelo(z); uzx=grad(uz,z,torch.ones_like(uz),create_graph=True)[0]
    uzxx=grad(uzx,z,torch.ones_like(uzx),create_graph=True)[0]
    Lv=-uzxx
    # Usamos todos los pares sin diagonal en el regularizador diferenciable.
    idx_i,idx_j=torch.triu_indices(len(z),len(z),offset=1,device=DEVICE)
    delta=(z[idx_i]-z[idx_j]).abs().pow(alpha)
    sq=((Lv[idx_i]-Lv[idx_j]).abs()/delta).square()
    indice_max = sq.detach().argmax()
    holder_sq=sq[indice_max]
    return jpde+jbc+peso*holder_sq,holder_sq


def entrenar_diagnostico_holder(n, metodo, alpha, seed, pasos,
                                 K_aux=48, muestreo='iid'):
    fijar_semilla(seed)
    modelo=RedPoissonRegularizada().to(DEVICE,dtype=DTYPE_REG)
    if muestreo=='iid':
        # Muestra iid uniformemente en [-1,1], ordenada solo para graficar.
        x_f=-1+2*torch.rand((n,1),device=DEVICE,dtype=DTYPE_REG)
        x_f=x_f.sort(dim=0).values
    else:
        x_f=rejilla_shin(n)
    x_b=torch.tensor([[-1.0],[1.0]],device=DEVICE,dtype=DTYPE_REG)
    # La malla auxiliar NO coincide con la de entrenamiento.
    x_aux=torch.linspace(-1,1,K_aux,device=DEVICE,dtype=DTYPE_REG).reshape(-1,1)
    optimizador=torch.optim.Adam(modelo.parameters(),lr=1e-3)
    for paso in range(pasos):
        optimizador.zero_grad(set_to_none=True)
        if metodo=='Holder':
            L,pen=perdida_holder_discreta(modelo,x_f,x_b,x_aux,alpha,
                                          peso=FACTOR_PESO_HOLDER * n**(-alpha-0.5))
        else:
            L,_,_,_=componentes_perdida(modelo,x_f,x_b,
                         'LIPR-Shin' if metodo=='LIPR' else 'PINN',n**(-1.5))
        L.backward()
        optimizador.step()
    return modelo,x_f.detach(),float(L.detach().cpu())


def radio_cobertura_1d(x_train):
    """Radio EXACTO en [-1,1] para puntos 1D ordenados."""
    s=np.sort(np.asarray(x_train).reshape(-1))
    return max(float(s[0]+1),float(1-s[-1]),
               float(np.diff(s).max()/2) if len(s)>1 else 0.)

print('Herramientas de Hölder, diagnóstico y cobertura preparadas.')

In [ ]:
# ----------------------------------------------------------
# Experimento Hölder: muestreo, residuo y errores independientes
# ----------------------------------------------------------
# Para no confundir este estudio con la figura de Shin, empleamos muestreo iid.
if MODO=='demostracion':
    N_HOLDER=[12,24]
    EPOCAS_HOLDER=20
    K_HOLDER=16
    N_EVAL_HOLDER=128
    MALLAS_HOLDER=[16,32,64]
else:
    N_HOLDER=[20,40,80,160]
    EPOCAS_HOLDER=550
    K_HOLDER=96
    N_EVAL_HOLDER=512
    MALLAS_HOLDER=[32,64,128,256,512]

filas_holder=[]
modelos_diagnostico={}
_x_evaluacion=np.linspace(-1,1,N_EVAL_HOLDER)
for numero in N_HOLDER:
    for metodo in ['PINN','LIPR','Holder']:
        modelo,x_train,perdida_entreno=entrenar_diagnostico_holder(
            numero,metodo,ALPHA_HOLDER,SEMILLA+17,EPOCAS_HOLDER,
            K_aux=K_HOLDER,muestreo='iid')
        estimacion_op=evaluar_operador(modelo,_x_evaluacion)
        _x_cuda=torch.as_tensor(_x_evaluacion,device=DEVICE,dtype=DTYPE_REG).reshape(-1,1)
        rhs=fuente_poisson_tanh(_x_cuda).detach().cpu().numpy().reshape(-1)
        residuo_eval=estimacion_op-rhs
        _x_f=x_train.cpu().numpy().reshape(-1)
        rhs_train=fuente_poisson_tanh(x_train).cpu().numpy().reshape(-1)
        residuo_train=evaluar_operador(modelo,_x_f)-rhs_train
        # Calculamos los errores respecto de la solución exacta en una malla de prueba.
        l2_rel,h1_rel=errores_relativos_1d(modelo,n_eval=1001)
        with torch.no_grad():
            u_eval=modelo(_x_cuda).cpu().numpy().reshape(-1)
        c0=float(np.max(np.abs(u_eval-np.tanh(_x_evaluacion))))
        delta=radio_cobertura_1d(_x_f)
        fila={'N_f':numero,'metodo':metodo,'muestreo':'iid',
              'alpha':ALPHA_HOLDER,'semilla':SEMILLA+17,
              'peso_holder_m':FACTOR_PESO_HOLDER * numero**(-ALPHA_HOLDER-0.5),
              'peso_lipr_m':numero**(-1.5),
              'perdida_ultimo_batch':perdida_entreno,
              'residuo_max_prueba':float(np.max(np.abs(residuo_eval))),
              'residuo_max_entrenamiento':float(np.max(np.abs(residuo_train))),
              'L2_rel':l2_rel,'H1_rel':h1_rel,'C0_evaluado':c0,
              'radio_cobertura_exacto':delta,'epocas':EPOCAS_HOLDER,
              'K_aux':K_HOLDER}
        filas_holder.append(fila)
        guardar_csv('resultados_diagnostico_holder.csv',filas_holder)
        if numero==max(N_HOLDER):
            modelos_diagnostico[metodo]=(modelo,x_train,_x_evaluacion,residuo_eval)
        print(f'Holder N={numero} {metodo:6}: L2={l2_rel:.2e}, '
              f'C0={c0:.2e}, residuo_max={fila["residuo_max_prueba"]:.2e}')

# Refinamiento de la malla auxiliar para calcular cotas inferiores S_K,alpha.
model_ref=modelos_diagnostico['LIPR'][0]
refinamiento_holder=[]
for K in MALLAS_HOLDER:
    z=np.linspace(-1,1,K)
    op=evaluar_operador(model_ref,z)
    px=torch.as_tensor(z.reshape(-1,1),dtype=DTYPE_REG)
    vx=torch.as_tensor(op.reshape(-1,1),dtype=DTYPE_REG)
    for alpha in [0.5,1.0]:
        S=aproximar_holder(px,vx,alpha)
        # Solo para alpha=1: por el teorema del valor medio, el supremo
        # continuo coincide con sup|L'[h]|, que también aproximamos.
        max_grad_malla = (float(np.max(evaluar_gradiente_operador_1d(model_ref,z)))
                          if alpha == 1.0 else np.nan)
        refinamiento_holder.append({'K':K,'alpha':alpha,
              'seminorma_discreta':S,'max_gradiente_operador_en_malla':max_grad_malla})
guardar_csv('refinamiento_seminorma_holder.csv',refinamiento_holder)

# Cubierta con puntos deterministas e iid, medida geométrica independiente del modelo.
filas_cobertura=[]
for n in [4,8,16,32,64,128]:
    rng=np.random.default_rng(SEMILLA+n)
    filas_cobertura.extend([
      {'N_f':n,'muestreo':'equiespaciado','radio_exacto':radio_cobertura_1d(np.linspace(-1,1,n))},
      {'N_f':n,'muestreo':'iid','radio_exacto':radio_cobertura_1d(rng.uniform(-1,1,n))},
    ])
guardar_csv('radios_cobertura_1d.csv',filas_cobertura)

In [ ]:
# ----------------------------------------------------------
# Figura: diagnóstico de la regularidad Hölder y de la cobertura
# ----------------------------------------------------------
fig,axes=plt.subplots(2,2,figsize=(14,9))

# A. Ejemplo matemático de pico no detectado por un muestreo escaso.
puntos_sparse=np.linspace(-1,1,10)
centro=(puntos_sparse[4]+puntos_sparse[5])/2
xx=np.linspace(-1,1,1200)
sigma=(puntos_sparse[5]-puntos_sparse[4])/9
pico=np.exp(-((xx-centro)/sigma)**2)
axes[0,0].plot(xx,pico,label='Residuo localizado (ejemplo construido)')
axes[0,0].scatter(puntos_sparse,np.exp(-((puntos_sparse-centro)/sigma)**2),
                  color='red',label='Puntos de colocación',zorder=5)
axes[0,0].set(title='A. Pico residual entre muestras',xlabel='x',ylabel='|r(x)|')
axes[0,0].legend(fontsize=8)

# B. Malla auxiliar y estimadores inferiores de la seminorma continua.
tabla_ref=pd.DataFrame(refinamiento_holder)
for alpha in [0.5,1.0]:
    d=tabla_ref[tabla_ref.alpha==alpha]
    axes[0,1].plot(d.K,d.seminorma_discreta,'o-',label=rf'$\alpha={alpha:g}$')
axes[0,1].set(title='B. Refinamiento de la seminorma Hölder discreta',
              xlabel='Número de puntos auxiliares K',ylabel=r'$S_{K,\alpha}$')
# Para alpha=1, comparación independiente con max|L'[h]| sobre Q_K.
d=tabla_ref[tabla_ref.alpha==1.0]
if d['max_gradiente_operador_en_malla'].notna().any():
    axes[0,1].plot(d.K, d.max_gradiente_operador_en_malla,'k--',
                   label=r'$\max_{Q_K}|\partial_x\mathcal{L}[h]|$')
axes[0,1].legend()

# C. Residuo máximo independiente versus número de muestras iid.
datos_h=pd.DataFrame(filas_holder)
for metodo in ['PINN','LIPR','Holder']:
    d=datos_h[datos_h.metodo==metodo].sort_values('N_f')
    axes[1,0].loglog(d.N_f,d.residuo_max_prueba,'o-',label=metodo)
axes[1,0].set(title='C. Residuo máximo en malla de prueba',
              xlabel=r'$N_f$ (muestras iid)',ylabel=r'$\max_{Q_K}|r_\theta|$')
axes[1,0].legend(fontsize=8)

# D. Error en C0 calculado frente al número de puntos.
for metodo in ['PINN','LIPR','Holder']:
    d=datos_h[datos_h.metodo==metodo].sort_values('N_f')
    axes[1,1].loglog(d.N_f,d.C0_evaluado,'o-',label=metodo)
axes[1,1].set(title='D. Error uniforme evaluado (no supremo certificado)',
              xlabel=r'$N_f$',ylabel=r'$\max_{Q_K}|h_\theta-u^*|$')
axes[1,1].legend(fontsize=8)
for ax in axes.flat:
    ax.grid(True,which='both',alpha=.25)
fig.suptitle('Regularidad Hölder, muestreo y error para Poisson 1D',fontsize=15)
guardar_figura('diagnostico_holder_poisson1d.png')

In [ ]:
# ----------------------------------------------------------
# Figura auxiliar: cobertura y prueba de la cota FINITA en la malla
# ----------------------------------------------------------
fig,axs=plt.subplots(1,2,figsize=(12.5,4.5))
cob=pd.DataFrame(filas_cobertura)
for modo in ['equiespaciado','iid']:
    d=cob[cob.muestreo==modo].sort_values('N_f')
    axs[0].loglog(d.N_f,d.radio_exacto,'o-',label=modo)
axs[0].set(xlabel=r'$N_f$',ylabel=r'$\delta_m$',
           title='Radio de cobertura exacto en [-1,1]')
axs[0].legend()

# Para una cuadrícula finita, la desigualdad es exacta si el máximo
# de cocientes se calcula sobre la UNIÓN de prueba y entrenamiento.
modelo,x_train,x_e,r_e=modelos_diagnostico['LIPR']
x_t=x_train.detach().cpu().numpy().reshape(-1)
union=np.unique(np.concatenate((x_e,x_t)))
operator=evaluar_operador(modelo,union)
_rhs=fuente_poisson_tanh(torch.tensor(union,device=DEVICE,dtype=DTYPE_REG).reshape(-1,1)).detach().cpu().numpy().reshape(-1)
z_t=torch.tensor(union.reshape(-1,1),dtype=DTYPE_REG)
for alpha in [0.5,1.0]:
    s_oper=aproximar_holder(z_t,torch.tensor(operator.reshape(-1,1)),alpha)
    s_rhs=aproximar_holder(z_t,torch.tensor(_rhs.reshape(-1,1)),alpha)
    t_res=evaluar_operador(modelo,x_t)-fuente_poisson_tanh(
      torch.tensor(x_t,device=DEVICE,dtype=DTYPE_REG).reshape(-1,1)).detach().cpu().numpy().reshape(-1)
    delta_grid=np.max(np.min(abs(x_e[:,None]-x_t[None,:]),axis=1))
    lado_izq=float(np.max(np.abs(r_e)))
    lado_der=float(np.max(np.abs(t_res))+(s_oper+s_rhs)*delta_grid**alpha)
    assert lado_izq <= lado_der+1e-8,('cota finita',alpha,lado_izq,lado_der)
    axs[1].bar([fr'LHS $\alpha={alpha}$',fr'RHS $\alpha={alpha}$'],[lado_izq,lado_der],
               alpha=.65,label=rf'$\alpha={alpha}$')
axs[1].set(title='Desigualdad de cobertura en malla FINITA',ylabel='Valor evaluado')
for ax in axs:ax.grid(True,alpha=.25)
fig.suptitle('Control geométrico del residuo: diagnóstico numérico')
guardar_figura('holder_cobertura_y_cota_finita.png')
print('La cota de la malla finita se verificó; NO implica una cota continua certificada.')

### Ejemplo IV: la ecuación de Poisson 2D

- Ahora queremos hallar una funcción $u(x,y)$ que satisface la siguiente EDP de  $2^\rm{nd}$ orden:

$$\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} = -2\pi^2 Sen(\pi x)Sen(\pi y)$$

- donde $(x,y) \in [-1,1]\times[-1,1]$.

- Además tenemos las siguientes condiciones de contorno:

$$u(-1,y)=0$$
$$u(1,y)=0$$
$$u(x,-1)=0$$
$$u(x,1)=0$$

##Solución con datos de observación

La solución exacta para comparar es $u^*(x,y)=\sin(\pi x)\sin(\pi y)$. En los experimentos con observaciones se utilizan los 25 puntos interiores de la cuadrícula $\{-0.8,-0.4,0,0.4,0.8\}^2$. También se penaliza el contorno homogéneo. Las derivadas se calculan por diferenciación automática en la GPU cuando esté disponible.

In [ ]:
# Poisson 2D con observaciones
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {
    'ep_poisson2d': 60 if MODO=='demostracion' else 6000,
    'n_poisson2d': 9 if MODO=='demostracion' else 75,
    'n_evaluacion': 25 if MODO=='demostracion' else 101,
}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Poisson 2D con observaciones:', DEVICE, '| salida:', SALIDA)

# ----------------------------------------------------------
# Poisson 2D: arquitectura, puntos y derivadas del operador
# ----------------------------------------------------------
def solucion_exacta_2d(p):
    return torch.sin(torch.pi * p[:, 0:1]) * torch.sin(torch.pi * p[:, 1:2])

def laplaciano_2d(modelo, puntos):
    # ∆u = u_xx + u_yy; el laplaciano no incluye derivadas mixtas.
    u = modelo(puntos)
    grads = torch.autograd.grad(u, puntos, torch.ones_like(u), create_graph=True)[0]
    ux = grads[:, 0:1]
    uy = grads[:, 1:2]
    uxx = torch.autograd.grad(ux, puntos, torch.ones_like(ux), create_graph=True)[0][:, 0:1]
    uyy = torch.autograd.grad(uy, puntos, torch.ones_like(uy), create_graph=True)[0][:, 1:2]
    return uxx + uyy

def crear_puntos_2d(N):
    # N subdivisiones por eje; los puntos residuales son interiores.
    eje = torch.linspace(-1, 1, N+1, device=DEVICE, dtype=DTYPE_2D)
    X, Y = torch.meshgrid(eje[1:-1], eje[1:-1], indexing='ij')
    interior = torch.stack((X.flatten(), Y.flatten()), dim=1)

    # Construimos explícitamente los cuatro lados de la frontera.
    lado_abajo = torch.stack((eje, torch.full_like(eje, -1)), dim=1)
    lado_arriba = torch.stack((eje, torch.full_like(eje, 1)), dim=1)
    lado_izq = torch.stack((torch.full_like(eje, -1), eje), dim=1)
    lado_der = torch.stack((torch.full_like(eje, 1), eje), dim=1)
    frontera = torch.cat((lado_abajo, lado_arriba, lado_izq, lado_der))
    return interior, frontera

class RedPoisson2D(nn.Module):
    def __init__(self, con_datos=False):
        super().__init__()
        # Arquitectura 40-80-40 en el caso con observaciones,
        # y 50-50 en la aproximación solo física.
        anchos = (40,80,40) if con_datos else (50,50)
        capas, anterior = [], 2
        for ancho in anchos:
            capas.extend([nn.Linear(anterior, ancho), nn.Tanh()])
            anterior = ancho
        capas.append(nn.Linear(anterior, 1))
        self.net = nn.Sequential(*capas)

    def forward(self, p):
        return self.net(p)

def entrenar_poisson_2d(N, epocas, con_datos, semilla):
    fijar_semilla(semilla)
    modelo = RedPoisson2D(con_datos).to(device=DEVICE, dtype=DTYPE_2D)
    interior, frontera = crear_puntos_2d(N)
    observaciones = None
    if con_datos:
        # Los mismos 25 puntos interiores del planteamiento original.
        eje_obs = torch.tensor([-.8, -.4, 0., .4, .8],
                               device=DEVICE, dtype=DTYPE_2D)
        Xo, Yo = torch.meshgrid(eje_obs, eje_obs, indexing='ij')
        observaciones = torch.stack((Xo.flatten(), Yo.flatten()), dim=1)

    optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    historial = []
    comienzo = time.perf_counter()
    for epoca in range(epocas):
        puntos = interior.detach().requires_grad_(True)
        # Ecuación ∆u = -2π² sin(πx) sin(πy).
        residuo = laplaciano_2d(modelo, puntos) + 2*torch.pi**2*solucion_exacta_2d(puntos)
        perdida_pde = residuo.square().mean()
        perdida_bc = modelo(frontera).square().mean()
        perdida_datos = ((modelo(observaciones)-solucion_exacta_2d(observaciones))
                         .square().mean() if con_datos else puntos.new_zeros(()))
        perdida = perdida_pde + 10.0*perdida_bc + perdida_datos
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()
        if epoca == 0 or (epoca+1) % max(1, epocas//20) == 0 or epoca+1 == epocas:
            historial.append({'epoca': epoca+1, 'total': float(perdida.detach()),
                              'pde': float(perdida_pde.detach()),
                              'bc': float(perdida_bc.detach()),
                              'datos': float(perdida_datos.detach())})
    return modelo, historial, time.perf_counter()-comienzo

def puntos_evaluacion_2d(N):
    # Cuadrícula interior común y fija para evaluar FEM, FVM y PINN.
    eje = np.linspace(-1,1,N+2)[1:-1]
    X,Y = np.meshgrid(eje,eje,indexing='ij')
    return np.column_stack((X.ravel(),Y.ravel()))

def predecir_2d(modelo, puntos, lote=5000):
    valores=[]
    for i in range(0,len(puntos),lote):
        p = torch.as_tensor(puntos[i:i+lote], device=DEVICE, dtype=DTYPE_2D)
        with torch.no_grad():
            valores.append(modelo(p).cpu().numpy().ravel())
    return np.concatenate(valores)

def error_l2_2d(prediccion, puntos):
    real = np.sin(np.pi*puntos[:,0])*np.sin(np.pi*puntos[:,1])
    prediccion=np.asarray(prediccion).reshape(-1)
    return float(np.linalg.norm(prediccion-real) / np.linalg.norm(real))

def error_h1_2d(modelo, puntos):
    p=torch.as_tensor(puntos,device=DEVICE,dtype=DTYPE_2D).requires_grad_(True)
    u=modelo(p)
    du=torch.autograd.grad(u,p,torch.ones_like(u))[0]
    exact=solucion_exacta_2d(p)
    x,y=p[:,0:1],p[:,1:2]
    du_exact=torch.cat((torch.pi*torch.cos(torch.pi*x)*torch.sin(torch.pi*y),
                        torch.pi*torch.sin(torch.pi*x)*torch.cos(torch.pi*y)),dim=1)
    error2=(u-exact).square().sum()+(du-du_exact).square().sum()
    norma2=exact.square().sum()+du_exact.square().sum()
    return float(torch.sqrt(error2/norma2).detach().cpu())

In [ ]:
# ----------------------------------------------------------
# Poisson 2D con observaciones interiores
# ----------------------------------------------------------
modelo_2d_datos, historial_2d_datos, segundos = entrenar_poisson_2d(
    CFG['n_poisson2d'], CFG['ep_poisson2d'], con_datos=True, semilla=SEMILLA+40
)
puntos_test_2d = puntos_evaluacion_2d(CFG['n_evaluacion'])
prediccion_2d_datos = predecir_2d(modelo_2d_datos, puntos_test_2d)
error_2d_datos = error_l2_2d(prediccion_2d_datos, puntos_test_2d)
error_2d_datos_h1 = error_h1_2d(modelo_2d_datos, puntos_test_2d)
print(f'Con observaciones: L2 = {error_2d_datos:.4e}, H1 = {error_2d_datos_h1:.4e}')
guardar_csv('historial_poisson2d_con_datos.csv', historial_2d_datos)

# Superficies de referencia y aproximación.
N_test=CFG['n_evaluacion']
X=puntos_test_2d[:,0].reshape(N_test,N_test)
Y=puntos_test_2d[:,1].reshape(N_test,N_test)
U=np.sin(np.pi*X)*np.sin(np.pi*Y)
P=prediccion_2d_datos.reshape(N_test,N_test)
fig=plt.figure(figsize=(12,5))
ax1=fig.add_subplot(121,projection='3d')
ax2=fig.add_subplot(122,projection='3d')
ax1.plot_surface(X,Y,U,cmap='viridis'); ax1.set_title('Solución exacta')
ax2.plot_surface(X,Y,P,cmap='viridis'); ax2.set_title('PINN con observaciones')
guardar_figura('poisson_2d_con_datos.png')
# Resultado numérico propio de esta sección (sin depender del caso sin datos).
guardar_csv('metricas_poisson2d_con_datos.csv', [{
    'caso': 'con datos', 'error_L2_rel': error_2d_datos,
    'error_H1_rel': error_2d_datos_h1, 'N_lado': CFG['n_poisson2d'],
    'epocas': CFG['ep_poisson2d']
}])


**Entrenamiento de Poisson 2D con observaciones.** Las superficies de la solución conservan la escala lineal. La gráfica siguiente muestra en **semilog** la pérdida total y sus componentes. El término de frontera se muestra multiplicado por 10, el mismo peso utilizado en la función objetivo.

Se incorpora una gráfica adicional de participación porcentual de EDP, frontera (con su peso $10$) y observaciones.

In [ ]:
# ----------------------------------------------------------
# Poisson 2D con datos: pérdidas total, residual y de frontera
# ----------------------------------------------------------
epocas = [fila['epoca'] for fila in historial_2d_datos]
plt.figure(figsize=(9, 5))

# Los valores almacenados no se transforman: solo cambia la escala del eje Y.
plt.semilogy(epocas, [fila['total'] for fila in historial_2d_datos],
             label='Pérdida total')
plt.semilogy(epocas, [fila['pde'] for fila in historial_2d_datos],
             label='Residuo de Poisson')
plt.semilogy(epocas, [10 * fila['bc'] for fila in historial_2d_datos],
             label='10 × frontera')
plt.semilogy(epocas, [fila['datos'] for fila in historial_2d_datos],
             label='Observaciones')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.title('Poisson 2D con observaciones: entrenamiento')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
guardar_figura('perdida_poisson_2d_con_datos.png')

# Participación de cada término EN LA PÉRDIDA TOTAL (no en el error L2).
# El factor 10 de Dirichlet está incluido en el porcentaje de frontera.
tabla_c = pd.DataFrame(historial_2d_datos)
componentes = np.column_stack([tabla_c['pde'], 10 * tabla_c['bc'], tabla_c['datos']])
total_c = componentes.sum(axis=1)
participacion = 100 * componentes / np.maximum(total_c[:, None], 1e-30)
fig, ax = plt.subplots(figsize=(9,4.5))
ax.stackplot(tabla_c['epoca'], *participacion.T,
             labels=['EDP', 'Frontera × 10', 'Datos'], alpha=.85)
ax.set(xlabel='Épocas', ylabel='Contribución a la pérdida (%)',
       ylim=(0,100), title='Poisson 2D: con observaciones')
ax.legend(loc='upper right')
ax.grid(alpha=.25)
guardar_figura('participacion_perdidas_poisson_2d_con_datos.png')


##Solución sin datos de observación interiores

In [ ]:
# Poisson 2D sin observaciones
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {
    'ep_poisson2d': 60 if MODO=='demostracion' else 6000,
    'n_poisson2d': 9 if MODO=='demostracion' else 75,
    'n_evaluacion': 25 if MODO=='demostracion' else 101,
}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Poisson 2D sin observaciones:', DEVICE, '| salida:', SALIDA)

# ----------------------------------------------------------
# Poisson 2D: arquitectura, puntos y derivadas del operador
# ----------------------------------------------------------
def solucion_exacta_2d(p):
    return torch.sin(torch.pi * p[:, 0:1]) * torch.sin(torch.pi * p[:, 1:2])

def laplaciano_2d(modelo, puntos):
    # ∆u = u_xx + u_yy; el laplaciano no incluye derivadas mixtas.
    u = modelo(puntos)
    grads = torch.autograd.grad(u, puntos, torch.ones_like(u), create_graph=True)[0]
    ux = grads[:, 0:1]
    uy = grads[:, 1:2]
    uxx = torch.autograd.grad(ux, puntos, torch.ones_like(ux), create_graph=True)[0][:, 0:1]
    uyy = torch.autograd.grad(uy, puntos, torch.ones_like(uy), create_graph=True)[0][:, 1:2]
    return uxx + uyy

def crear_puntos_2d(N):
    # N subdivisiones por eje; los puntos residuales son interiores.
    eje = torch.linspace(-1, 1, N+1, device=DEVICE, dtype=DTYPE_2D)
    X, Y = torch.meshgrid(eje[1:-1], eje[1:-1], indexing='ij')
    interior = torch.stack((X.flatten(), Y.flatten()), dim=1)

    # Construimos explícitamente los cuatro lados de la frontera.
    lado_abajo = torch.stack((eje, torch.full_like(eje, -1)), dim=1)
    lado_arriba = torch.stack((eje, torch.full_like(eje, 1)), dim=1)
    lado_izq = torch.stack((torch.full_like(eje, -1), eje), dim=1)
    lado_der = torch.stack((torch.full_like(eje, 1), eje), dim=1)
    frontera = torch.cat((lado_abajo, lado_arriba, lado_izq, lado_der))
    return interior, frontera

class RedPoisson2D(nn.Module):
    def __init__(self, con_datos=False):
        super().__init__()
        # Arquitectura 40-80-40 en el caso con observaciones,
        # y 50-50 en la aproximación solo física.
        anchos = (40,80,40) if con_datos else (50,50)
        capas, anterior = [], 2
        for ancho in anchos:
            capas.extend([nn.Linear(anterior, ancho), nn.Tanh()])
            anterior = ancho
        capas.append(nn.Linear(anterior, 1))
        self.net = nn.Sequential(*capas)

    def forward(self, p):
        return self.net(p)

def entrenar_poisson_2d(N, epocas, con_datos, semilla):
    fijar_semilla(semilla)
    modelo = RedPoisson2D(con_datos).to(device=DEVICE, dtype=DTYPE_2D)
    interior, frontera = crear_puntos_2d(N)
    observaciones = None
    if con_datos:
        # Los mismos 25 puntos interiores del planteamiento original.
        eje_obs = torch.tensor([-.8, -.4, 0., .4, .8],
                               device=DEVICE, dtype=DTYPE_2D)
        Xo, Yo = torch.meshgrid(eje_obs, eje_obs, indexing='ij')
        observaciones = torch.stack((Xo.flatten(), Yo.flatten()), dim=1)

    optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    historial = []
    comienzo = time.perf_counter()
    for epoca in range(epocas):
        puntos = interior.detach().requires_grad_(True)
        # Ecuación ∆u = -2π² sin(πx) sin(πy).
        residuo = laplaciano_2d(modelo, puntos) + 2*torch.pi**2*solucion_exacta_2d(puntos)
        perdida_pde = residuo.square().mean()
        perdida_bc = modelo(frontera).square().mean()
        perdida_datos = ((modelo(observaciones)-solucion_exacta_2d(observaciones))
                         .square().mean() if con_datos else puntos.new_zeros(()))
        perdida = perdida_pde + 10.0*perdida_bc + perdida_datos
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()
        if epoca == 0 or (epoca+1) % max(1, epocas//20) == 0 or epoca+1 == epocas:
            historial.append({'epoca': epoca+1, 'total': float(perdida.detach()),
                              'pde': float(perdida_pde.detach()),
                              'bc': float(perdida_bc.detach()),
                              'datos': float(perdida_datos.detach())})
    return modelo, historial, time.perf_counter()-comienzo

def puntos_evaluacion_2d(N):
    # Cuadrícula interior común y fija para evaluar FEM, FVM y PINN.
    eje = np.linspace(-1,1,N+2)[1:-1]
    X,Y = np.meshgrid(eje,eje,indexing='ij')
    return np.column_stack((X.ravel(),Y.ravel()))

def predecir_2d(modelo, puntos, lote=5000):
    valores=[]
    for i in range(0,len(puntos),lote):
        p = torch.as_tensor(puntos[i:i+lote], device=DEVICE, dtype=DTYPE_2D)
        with torch.no_grad():
            valores.append(modelo(p).cpu().numpy().ravel())
    return np.concatenate(valores)

def error_l2_2d(prediccion, puntos):
    real = np.sin(np.pi*puntos[:,0])*np.sin(np.pi*puntos[:,1])
    prediccion=np.asarray(prediccion).reshape(-1)
    return float(np.linalg.norm(prediccion-real) / np.linalg.norm(real))

def error_h1_2d(modelo, puntos):
    p=torch.as_tensor(puntos,device=DEVICE,dtype=DTYPE_2D).requires_grad_(True)
    u=modelo(p)
    du=torch.autograd.grad(u,p,torch.ones_like(u))[0]
    exact=solucion_exacta_2d(p)
    x,y=p[:,0:1],p[:,1:2]
    du_exact=torch.cat((torch.pi*torch.cos(torch.pi*x)*torch.sin(torch.pi*y),
                        torch.pi*torch.sin(torch.pi*x)*torch.cos(torch.pi*y)),dim=1)
    error2=(u-exact).square().sum()+(du-du_exact).square().sum()
    norma2=exact.square().sum()+du_exact.square().sum()
    return float(torch.sqrt(error2/norma2).detach().cpu())
# ----------------------------------------------------------
# Poisson 2D sin observaciones interiores
# ----------------------------------------------------------
modelo_2d_sin, historial_2d_sin, segundos = entrenar_poisson_2d(
    CFG['n_poisson2d'], CFG['ep_poisson2d'], con_datos=False, semilla=SEMILLA+41
)
puntos_test_2d = puntos_evaluacion_2d(CFG['n_evaluacion'])
N_test = CFG['n_evaluacion']
X = puntos_test_2d[:, 0].reshape(N_test, N_test)
Y = puntos_test_2d[:, 1].reshape(N_test, N_test)
U = np.sin(np.pi*X)*np.sin(np.pi*Y)
prediccion_2d_sin = predecir_2d(modelo_2d_sin, puntos_test_2d)
error_2d_sin = error_l2_2d(prediccion_2d_sin, puntos_test_2d)
error_2d_sin_h1 = error_h1_2d(modelo_2d_sin, puntos_test_2d)
print(f'Sin observaciones: L2 = {error_2d_sin:.4e}, H1 = {error_2d_sin_h1:.4e}')
guardar_csv('historial_poisson2d_sin_datos.csv', historial_2d_sin)

guardar_csv('metricas_poisson2d_sin_datos.csv', [
    {'caso':'sin datos', 'error_L2_rel':error_2d_sin, 'error_H1_rel':error_2d_sin_h1,
     'N_lado':CFG['n_poisson2d'], 'epocas':CFG['ep_poisson2d']},
])
# La comparación se construye desde archivos, sin depender de otro modelo en memoria.
metricas_partes = []
for nombre in ['metricas_poisson2d_con_datos.csv', 'metricas_poisson2d_sin_datos.csv']:
    if (SALIDA/nombre).exists():
        metricas_partes.append(pd.read_csv(SALIDA/nombre))
if metricas_partes:
    pd.concat(metricas_partes, ignore_index=True).to_csv(SALIDA/'metricas_poisson2d.csv', index=False)

P=prediccion_2d_sin.reshape(N_test,N_test)
fig=plt.figure(figsize=(12,5))
ax1=fig.add_subplot(121,projection='3d')
ax2=fig.add_subplot(122,projection='3d')
ax1.plot_surface(X,Y,U,cmap='viridis'); ax1.set_title('Solución exacta')
ax2.plot_surface(X,Y,P,cmap='viridis'); ax2.set_title('PINN sin observaciones')
guardar_figura('poisson_2d_sin_datos.png')

**Entrenamiento de Poisson 2D sin observaciones interiores.** Se representa la pérdida total, el residuo y el término de frontera en **semilog**. No se grafica una pérdida de datos, porque en este experimento no existe ese término.

Se incorpora la participación porcentual del residuo y de la frontera con peso $10$.

In [ ]:
# ----------------------------------------------------------
# Poisson 2D sin datos: pérdida residual y condición de Dirichlet
# ----------------------------------------------------------
epocas = [fila['epoca'] for fila in historial_2d_sin]
plt.figure(figsize=(9, 5))

plt.semilogy(epocas, [fila['total'] for fila in historial_2d_sin],
             label='Pérdida total')
plt.semilogy(epocas, [fila['pde'] for fila in historial_2d_sin],
             label='Residuo de Poisson')
plt.semilogy(epocas, [10 * fila['bc'] for fila in historial_2d_sin],
             label='10 × frontera')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.title('Poisson 2D sin observaciones: entrenamiento')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
guardar_figura('perdida_poisson_2d_sin_datos.png')

# Participación de cada término EN LA PÉRDIDA TOTAL (no en el error L2).
# El factor 10 de Dirichlet está incluido en el porcentaje de frontera.
tabla_c = pd.DataFrame(historial_2d_sin)
componentes = np.column_stack([tabla_c['pde'], 10 * tabla_c['bc']])
total_c = componentes.sum(axis=1)
participacion = 100 * componentes / np.maximum(total_c[:, None], 1e-30)
fig, ax = plt.subplots(figsize=(9,4.5))
ax.stackplot(tabla_c['epoca'], *participacion.T,
             labels=['EDP', 'Frontera × 10'], alpha=.85)
ax.set(xlabel='Épocas', ylabel='Contribución a la pérdida (%)',
       ylim=(0,100), title='Poisson 2D: sin observaciones')
ax.legend(loc='upper right')
ax.grid(alpha=.25)
guardar_figura('participacion_perdidas_poisson_2d_sin_datos.png')


# **Usando un solver**

## Comparación de los métodos FEM, FVM y PINN

La misma ecuación $\Delta u=-2\pi^2\sin(\pi x)\sin(\pi y)$ se resuelve con **scikit-fem** (elementos finitos), **FiPy** (volúmenes finitos) y una PINN sin observaciones. Los tres métodos se evalúan en una **cuadrícula interior de prueba común**, independiente de los puntos empleados durante sus cálculos.

El parámetro $N$ designa subdivisiones por dirección en FEM/FVM y densidad de puntos de colocación para la PINN. **No representa un costo computacional comparable** entre los métodos. La tabla guarda también el tiempo de cálculo para distinguir precisión y costo.




In [ ]:
# Poisson 2D: FEM, FVM y PINN
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {
    'n_benchmark': [5,9] if MODO=='demostracion' else [10,20,30,40,60,80],
    'ep_benchmark_pinn': 60 if MODO=='demostracion' else 4000,
    'n_evaluacion': 25 if MODO=='demostracion' else 101,
}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Comparación FEM / FVM / PINN:', DEVICE, '| salida:', SALIDA)

# ----------------------------------------------------------
# Poisson 2D: arquitectura, puntos y derivadas del operador
# ----------------------------------------------------------
def solucion_exacta_2d(p):
    return torch.sin(torch.pi * p[:, 0:1]) * torch.sin(torch.pi * p[:, 1:2])

def laplaciano_2d(modelo, puntos):
    # ∆u = u_xx + u_yy; el laplaciano no incluye derivadas mixtas.
    u = modelo(puntos)
    grads = torch.autograd.grad(u, puntos, torch.ones_like(u), create_graph=True)[0]
    ux = grads[:, 0:1]
    uy = grads[:, 1:2]
    uxx = torch.autograd.grad(ux, puntos, torch.ones_like(ux), create_graph=True)[0][:, 0:1]
    uyy = torch.autograd.grad(uy, puntos, torch.ones_like(uy), create_graph=True)[0][:, 1:2]
    return uxx + uyy

def crear_puntos_2d(N):
    # N subdivisiones por eje; los puntos residuales son interiores.
    eje = torch.linspace(-1, 1, N+1, device=DEVICE, dtype=DTYPE_2D)
    X, Y = torch.meshgrid(eje[1:-1], eje[1:-1], indexing='ij')
    interior = torch.stack((X.flatten(), Y.flatten()), dim=1)

    # Construimos explícitamente los cuatro lados de la frontera.
    lado_abajo = torch.stack((eje, torch.full_like(eje, -1)), dim=1)
    lado_arriba = torch.stack((eje, torch.full_like(eje, 1)), dim=1)
    lado_izq = torch.stack((torch.full_like(eje, -1), eje), dim=1)
    lado_der = torch.stack((torch.full_like(eje, 1), eje), dim=1)
    frontera = torch.cat((lado_abajo, lado_arriba, lado_izq, lado_der))
    return interior, frontera

class RedPoisson2D(nn.Module):
    def __init__(self, con_datos=False):
        super().__init__()
        # Arquitectura 40-80-40 en el caso con observaciones,
        # y 50-50 en la aproximación solo física.
        anchos = (40,80,40) if con_datos else (50,50)
        capas, anterior = [], 2
        for ancho in anchos:
            capas.extend([nn.Linear(anterior, ancho), nn.Tanh()])
            anterior = ancho
        capas.append(nn.Linear(anterior, 1))
        self.net = nn.Sequential(*capas)

    def forward(self, p):
        return self.net(p)

def entrenar_poisson_2d(N, epocas, con_datos, semilla):
    fijar_semilla(semilla)
    modelo = RedPoisson2D(con_datos).to(device=DEVICE, dtype=DTYPE_2D)
    interior, frontera = crear_puntos_2d(N)
    observaciones = None
    if con_datos:
        # Los mismos 25 puntos interiores del planteamiento original.
        eje_obs = torch.tensor([-.8, -.4, 0., .4, .8],
                               device=DEVICE, dtype=DTYPE_2D)
        Xo, Yo = torch.meshgrid(eje_obs, eje_obs, indexing='ij')
        observaciones = torch.stack((Xo.flatten(), Yo.flatten()), dim=1)

    optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    historial = []
    comienzo = time.perf_counter()
    for epoca in range(epocas):
        puntos = interior.detach().requires_grad_(True)
        # Ecuación ∆u = -2π² sin(πx) sin(πy).
        residuo = laplaciano_2d(modelo, puntos) + 2*torch.pi**2*solucion_exacta_2d(puntos)
        perdida_pde = residuo.square().mean()
        perdida_bc = modelo(frontera).square().mean()
        perdida_datos = ((modelo(observaciones)-solucion_exacta_2d(observaciones))
                         .square().mean() if con_datos else puntos.new_zeros(()))
        perdida = perdida_pde + 10.0*perdida_bc + perdida_datos
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()
        if epoca == 0 or (epoca+1) % max(1, epocas//20) == 0 or epoca+1 == epocas:
            historial.append({'epoca': epoca+1, 'total': float(perdida.detach()),
                              'pde': float(perdida_pde.detach()),
                              'bc': float(perdida_bc.detach()),
                              'datos': float(perdida_datos.detach())})
    return modelo, historial, time.perf_counter()-comienzo

def puntos_evaluacion_2d(N):
    # Cuadrícula interior común y fija para evaluar FEM, FVM y PINN.
    eje = np.linspace(-1,1,N+2)[1:-1]
    X,Y = np.meshgrid(eje,eje,indexing='ij')
    return np.column_stack((X.ravel(),Y.ravel()))

def predecir_2d(modelo, puntos, lote=5000):
    valores=[]
    for i in range(0,len(puntos),lote):
        p = torch.as_tensor(puntos[i:i+lote], device=DEVICE, dtype=DTYPE_2D)
        with torch.no_grad():
            valores.append(modelo(p).cpu().numpy().ravel())
    return np.concatenate(valores)

def error_l2_2d(prediccion, puntos):
    real = np.sin(np.pi*puntos[:,0])*np.sin(np.pi*puntos[:,1])
    prediccion=np.asarray(prediccion).reshape(-1)
    return float(np.linalg.norm(prediccion-real) / np.linalg.norm(real))

def error_h1_2d(modelo, puntos):
    p=torch.as_tensor(puntos,device=DEVICE,dtype=DTYPE_2D).requires_grad_(True)
    u=modelo(p)
    du=torch.autograd.grad(u,p,torch.ones_like(u))[0]
    exact=solucion_exacta_2d(p)
    x,y=p[:,0:1],p[:,1:2]
    du_exact=torch.cat((torch.pi*torch.cos(torch.pi*x)*torch.sin(torch.pi*y),
                        torch.pi*torch.sin(torch.pi*x)*torch.cos(torch.pi*y)),dim=1)
    error2=(u-exact).square().sum()+(du-du_exact).square().sum()
    norma2=exact.square().sum()+du_exact.square().sum()
    return float(torch.sqrt(error2/norma2).detach().cpu())
# ----------------------------------------------------------
# Dependencias adicionales de los métodos tradicionales
# ----------------------------------------------------------
# En Colab se instalan una sola vez; en un entorno local con ambas
# bibliotecas presentes no se modifica la instalación de Python.
paquetes = [('skfem','scikit-fem'), ('fipy','fipy')]
faltantes=[nombre for modulo,nombre in paquetes if importlib.util.find_spec(modulo) is None]
if faltantes:
    print('Instalando:', ', '.join(faltantes))
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*faltantes])

import fipy as fp
from skfem import MeshQuad, Basis, ElementQuad1, BilinearForm, LinearForm, asm, condense, solve
from skfem.models.poisson import laplace

print('FiPy y scikit-fem disponibles.')

In [ ]:
# ----------------------------------------------------------
# Método de Elementos Finitos FEM: scikit-fem
# ----------------------------------------------------------
def resolver_fem(N, puntos_test):
    # Malla cuadrilateral sobre [-1,1]^2 con N celdas por dirección.
    eje = np.linspace(-1.0, 1.0, N+1)
    malla = MeshQuad.init_tensor(eje, eje)
    base = Basis(malla, ElementQuad1())

    # Formulación débil: ∫ grad(u)·grad(v) = ∫ (2π²sinπx sinπy) v.
    @LinearForm
    def carga(v, w):
        x,y = w.x[0], w.x[1]
        return (2*np.pi**2*np.sin(np.pi*x)*np.sin(np.pi*y))*v

    A = asm(laplace, base)
    b = asm(carga, base)
    coeficientes = solve(*condense(A, b, D=base.get_dofs()))

    # Evaluar la solución FEM en los mismos puntos que los demás métodos.
    return np.asarray(base.probes(puntos_test.T) @ coeficientes).reshape(-1)

# ----------------------------------------------------------
# Método de Volúmenes Finitos FVM: FiPy
# ----------------------------------------------------------
def resolver_fvm(N, puntos_test):
    paso = 2.0/N
    # Grid2D genera inicialmente coordenadas en [0,2]^2.
    malla = fp.Grid2D(dx=paso, dy=paso, nx=N, ny=N)
    xc = np.asarray(malla.cellCenters[0])-1.0
    yc = np.asarray(malla.cellCenters[1])-1.0
    fuente = -2.0*np.pi**2*np.sin(np.pi*xc)*np.sin(np.pi*yc)

    # Imponer Dirichlet homogénea sobre TODAS las caras externas.
    phi = fp.CellVariable(mesh=malla, value=0.0)
    phi.constrain(0.0, where=malla.exteriorFaces)
    rhs = fp.CellVariable(mesh=malla, value=fuente)
    (fp.DiffusionTerm() == rhs).solve(var=phi)

    # Las incógnitas de FiPy están en los CENTROS de las celdas.
    # Se interpolan hacia la misma malla de prueba que FEM y PINN.
    centros = -1.0+(np.arange(N)+0.5)*paso
    coordenadas = np.concatenate(([-1.0],centros,[1.0]))
    valores = np.zeros((N+2,N+2))
    valores[1:-1,1:-1] = np.asarray(phi.value).reshape((N,N),order='F')
    interpolador = RegularGridInterpolator((coordenadas,coordenadas),valores)
    return interpolador(puntos_test)

In [ ]:
# ----------------------------------------------------------
# Barrido de resolución para FEM, FVM y PINN
# ----------------------------------------------------------
resultados_comparacion = []
# Se conservan las tres predicciones en una resolución de referencia.
# Así todas las superficies y errores se calculan con la misma cuadrícula.
N_FIGURA = max(CFG['n_benchmark'])
campos_benchmark = {}
puntos_comunes = puntos_evaluacion_2d(CFG['n_evaluacion'])

for N in CFG['n_benchmark']:
    print(f'\nResolución N={N}')

    # FEM y FVM: se resuelven directamente con sus propios solvers.
    for metodo, solver in [('FEM',resolver_fem), ('FVM',resolver_fvm)]:
        inicio=time.perf_counter()
        estimacion=solver(N, puntos_comunes)
        segundos=time.perf_counter()-inicio
        error=error_l2_2d(estimacion,puntos_comunes)
        resultados_comparacion.append({
            'metodo':metodo,'N':N,'error_L2_rel':error,'segundos':segundos,
            'puntos_evaluacion':len(puntos_comunes),'protocolo':MODO
        })
        if N == N_FIGURA:
            campos_benchmark[metodo] = np.asarray(estimacion, dtype=float).copy()
        guardar_csv('comparacion_fem_fvm_pinn.csv',resultados_comparacion)
        print(f'  {metodo}: error relativo L2={error:.4e}')

    # PINN: el mismo problema físico, sin datos de observación interiores.
    red_pinn, historial, segundos = entrenar_poisson_2d(
        N, CFG['ep_benchmark_pinn'], con_datos=False, semilla=SEMILLA+100+N
    )
    estimacion=predecir_2d(red_pinn, puntos_comunes)
    error=error_l2_2d(estimacion,puntos_comunes)
    resultados_comparacion.append({
        'metodo':'PINN','N':N,'error_L2_rel':error,'segundos':segundos,
        'puntos_evaluacion':len(puntos_comunes),'protocolo':MODO
    })
    if N == N_FIGURA:
        campos_benchmark['PINN'] = np.asarray(estimacion, dtype=float).copy()
    guardar_csv('comparacion_fem_fvm_pinn.csv',resultados_comparacion)
    print(f'  PINN: error relativo L2={error:.4e}')

**Escala de la comparación.** Los errores relativos $L^2$ se grafican en **log-log** porque se estudian frente al parámetro de refinamiento $N$. Para interpretar una pendiente como tasa de convergencia deben observarse suficientes resoluciones dentro de un régimen aproximadamente lineal en escala logarítmica. No se identifica $N$ con el costo computacional ni se atribuyen tasas asintóticas demostradas a las curvas PINN.

In [ ]:
# ----------------------------------------------------------
# Gráfica log-log: errores calculados (sin tablas fijas)
# ----------------------------------------------------------
datos = pd.DataFrame(resultados_comparacion)
plt.figure(figsize=(9,6))
for metodo,simbolo in [('FEM','o-'),('FVM','s-'),('PINN','d--')]:
    d=datos[datos['metodo']==metodo].sort_values('N')
    plt.loglog(d['N'],d['error_L2_rel'],simbolo,linewidth=2,label=metodo)
plt.xlabel('Parámetro de resolución $N$')
plt.ylabel('Error relativo $L^2$ en cuadrícula de evaluación común')
plt.title('Comparación de FEM, FVM y PINN para Poisson 2D')
plt.legend()
plt.grid(True,which='both',alpha=.35)
guardar_figura('figura_4_9_comparacion_metodos.png')

# Tabla numérica con el error y el tiempo de cada cálculo.
print(datos.to_string(index=False))

### Comparación visual de superficies y distribución espacial del error

Se representa el **mismo problema de Poisson 2D**, con solución exacta $u^*(x,y)=\sin(\pi x)\sin(\pi y)$, en la resolución $N$ mayor del barrido anterior. En la primera figura aparecen, como en una comparación visual de soluciones numéricas, la superficie analítica, la obtenida mediante PINN y la calculada con FiPy (FVM), además de sus **errores absolutos puntuales** $|u_{\mathrm{aprox}}-u^*|$.

Para evitar una interpretación engañosa, las superficies de solución comparten los mismos límites verticales y las dos superficies de error comparten **la misma escala vertical**. Se agrega una visualización espacial de errores para **FEM, FVM y PINN**, con una escala de color común, y una figura de errores globales relativos $L^2$ y máximos absolutos en esa resolución.

Estas figuras complementan —y no sustituyen— la comparación **log-log** de errores frente a $N$. Todas se generan directamente a partir de las predicciones guardadas por el barrido: no se vuelven a entrenar las redes ni se cargan arreglos de errores precalculados. Los errores globales se calculan en una cuadrícula interior común, por lo que representan errores **discretos de evaluación**, no integrales continuas exactas.


In [ ]:
# ----------------------------------------------------------
# Superficies 3D: solución exacta, PINN, FiPy y sus errores
# ----------------------------------------------------------
# Los campos fueron guardados al realizar el barrido comparativo.
# NO se realiza un segundo entrenamiento para elaborar estas figuras.
assert set(campos_benchmark) == {'FEM', 'FVM', 'PINN'}, (
    'Ejecute primero el barrido FEM/FVM/PINN para obtener las tres predicciones.'
)

numero_puntos = len(puntos_comunes)
n_lado = int(np.sqrt(numero_puntos))
assert n_lado * n_lado == numero_puntos, 'La cuadrícula debe ser cuadrada.'

# Misma cuadrícula y solución analítica para los tres métodos.
X_comun = puntos_comunes[:, 0].reshape(n_lado, n_lado)
Y_comun = puntos_comunes[:, 1].reshape(n_lado, n_lado)
U_exacta = np.sin(np.pi * X_comun) * np.sin(np.pi * Y_comun)
U_pinn = campos_benchmark['PINN'].reshape(n_lado, n_lado)
U_fipy = campos_benchmark['FVM'].reshape(n_lado, n_lado)
U_fem = campos_benchmark['FEM'].reshape(n_lado, n_lado)

# Errores ABSOLUTOS (puntuales): magnitudes siempre no negativas.
error_pinn_abs = np.abs(U_pinn - U_exacta)
error_fipy_abs = np.abs(U_fipy - U_exacta)
error_fem_abs = np.abs(U_fem - U_exacta)

# Escalas idénticas para comparar las alturas y el color entre superficies.
limite_solucion = max(1.0, float(np.max(np.abs(U_exacta))),
                      float(np.max(np.abs(U_pinn))), float(np.max(np.abs(U_fipy))))
limite_error = max(1.e-12, float(error_pinn_abs.max()), float(error_fipy_abs.max()))

fig = plt.figure(figsize=(18, 10))
superficies = [
    ('Solución analítica', U_exacta),
    ('Solución PINN', U_pinn),
    ('Solución FiPy (FVM)', U_fipy),
]
for posicion, (titulo, valores) in enumerate(superficies, start=1):
    ax = fig.add_subplot(2, 3, posicion, projection='3d')
    ax.plot_surface(X_comun, Y_comun, valores, cmap='viridis',
                    vmin=-limite_solucion, vmax=limite_solucion,
                    linewidth=0, antialiased=True, rcount=55, ccount=55)
    ax.set_zlim(-limite_solucion, limite_solucion)
    ax.set_title(titulo)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('u(x,y)')

for posicion, (titulo, valores) in enumerate([
    ('Error absoluto PINN', error_pinn_abs),
    ('Error absoluto FiPy', error_fipy_abs),
], start=4):
    ax = fig.add_subplot(2, 3, posicion, projection='3d')
    ax.plot_surface(X_comun, Y_comun, valores, cmap='magma',
                    vmin=0.0, vmax=limite_error,
                    linewidth=0, antialiased=True, rcount=55, ccount=55)
    ax.set_zlim(0, limite_error)
    ax.set_title(titulo)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('|error|')

# El último panel resume el experimento sin mezclar escalas.
ax = fig.add_subplot(2, 3, 6)
ax.axis('off')
ax.text(0.04, 0.83, 'Problema de Poisson 2D', fontsize=13, weight='bold')
ax.text(0.04, 0.65, f'Resolución del barrido: N = {N_FIGURA}', fontsize=12)
ax.text(0.04, 0.49, f'Puntos de evaluación: {numero_puntos}', fontsize=12)
ax.text(0.04, 0.33, 'Error mostrado: |aproximación - exacta|', fontsize=11)
ax.text(0.04, 0.17, 'Escala vertical común para los dos errores', fontsize=10)
fig.suptitle('Poisson 2D: soluciones y errores puntuales (misma cuadrícula)', fontsize=16)
guardar_figura('comparacion_3d_soluciones_errores_Poisson2D.png')


In [ ]:
# ----------------------------------------------------------
# Comparación espacial y cuantitativa del error de los tres métodos
# ----------------------------------------------------------
from matplotlib.colors import Normalize

campos = {'FEM': U_fem, 'FVM (FiPy)': U_fipy, 'PINN': U_pinn}
errores = {nombre: np.abs(campo - U_exacta)
           for nombre, campo in campos.items()}
max_comun = max(1.e-12, *(float(e.max()) for e in errores.values()))

# A) Mapas de error absoluto: una única escala de color para FEM/FVM/PINN.
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True, sharey=True)
for ax, (nombre, e_abs) in zip(ejes, errores.items()):
    im = ax.imshow(e_abs.T, origin='lower', extent=(-1,1,-1,1),
                   vmin=0, vmax=max_comun, cmap='magma', aspect='equal')
    ax.set_title(f'|Error|: {nombre}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
fig.colorbar(im, ax=ejes, label='Error absoluto puntual', fraction=0.03, pad=0.02)
fig.suptitle(f'Distribución espacial del error, N={N_FIGURA}')
# Evitamos tight_layout incompatible con el colorbar común de tres ejes.
fig.subplots_adjust(top=0.82, right=0.89, wspace=0.18)
plt.savefig(SALIDA/'errores_espaciales_FEM_FVM_PINN.png', dpi=180, bbox_inches='tight')
plt.show(); plt.close(fig)

# B) Tabla de errores sobre la misma cuadrícula interior de prueba.
# L2 relativo = ||u_aprox - u*||_2 / ||u*||_2 (norma discreta).
# L-infinito absoluto = máximo puntual de |u_aprox-u*|.
filas_metricas = []
for nombre, campo in campos.items():
    diferencia = campo - U_exacta
    filas_metricas.append({
        'metodo': nombre,
        'N': N_FIGURA,
        'error_L2_rel': float(np.linalg.norm(diferencia.ravel()) /
                              np.linalg.norm(U_exacta.ravel())),
        'error_max_abs': float(np.max(np.abs(diferencia))),
        'error_RMSE': float(np.sqrt(np.mean(diferencia**2))),
        'puntos_evaluacion': numero_puntos,
    })
metricas_visuales = guardar_csv('errores_comparados_2d.csv', filas_metricas)
print(metricas_visuales.to_string(index=False))

# C) Comparación de las dos métricas, sin atribuirlas al costo de cálculo.
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, columna, titulo in zip(
    ejes,
    ['error_L2_rel', 'error_max_abs'],
    [r'Error relativo $L^2$', r'Error absoluto máximo $L^\infty$'],
):
    ax.bar(metricas_visuales['metodo'], metricas_visuales[columna],
           color=['#4C78A8','#F58518','#54A24B'])
    ax.set_yscale('log')
    ax.set_title(titulo)
    ax.set_ylabel('Error (escala logarítmica)')
    ax.grid(True, axis='y', which='both', alpha=0.3)
    ax.set_axisbelow(True)
fig.suptitle(f'FEM, FiPy y PINN: comparación de errores para N={N_FIGURA}')
guardar_figura('errores_globales_comparados_2d.png')


## Imágenes para incorporar al capítulo IV en LaTeX

Las gráficas se guardan con nombres consistentes con los argumentos `\includegraphics` del capítulo. Las imágenes provienen de **estas ejecuciones**, no de arreglos de errores previamente guardados. Cuando se utiliza un protocolo acotado, la etiqueta de la figura de Shin indica tal condición; para contrastar cuantitativamente con los autores debe ejecutarse el modo `articulo`.

In [ ]:
# Preparación de figuras y resultados
# Bibliotecas y parámetros propios del experimento.
import os
import sys
import json
import time
import random
import subprocess
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

# Parámetros de la ejecución.
MODO = 'experimento'  # 'demostracion' para una ejecución corta
SEMILLA = 2026
GUARDAR_EN_DRIVE = True  # resultados persistentes en Google Drive
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE_REG = torch.float64
DTYPE_2D = torch.float32
CFG = {}

def fijar_semilla(semilla):
    random.seed(int(semilla))
    np.random.seed(int(semilla))
    torch.manual_seed(int(semilla))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(semilla))

fijar_semilla(SEMILLA)

if GUARDAR_EN_DRIVE and importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    SALIDA = Path('/content/drive/MyDrive/Experimentos_PINN_PUCP') / MODO
else:
    SALIDA = Path('resultados_pinn') / MODO
SALIDA.mkdir(parents=True, exist_ok=True)

def guardar_csv(nombre, filas):
    tabla = pd.DataFrame(filas)
    if len(tabla):
        tabla.to_csv(SALIDA / nombre, index=False)
    return tabla

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

print('Figuras y archivos:', DEVICE, '| salida:', SALIDA)

# Recopilación: depende solamente de archivos PNG/CSV previamente guardados.
# Para copiar figuras de diferentes ubicaciones se comprueban ambas rutas.
RUTAS_FIGURAS = [SALIDA]
if Path('/content/drive/MyDrive/Experimentos_PINN_PUCP').exists():
    RUTAS_FIGURAS.append(Path('/content/drive/MyDrive/Experimentos_PINN_PUCP')/MODO)

# ----------------------------------------------------------
# Preparar automáticamente todos los PNG que usa el capítulo LaTeX
# ----------------------------------------------------------
import shutil
nombres_figura = {
    # Imágenes introductorias: el código original las genera en cada sección.
    'regresion_simple.png': 'imagen2.png',
    'solucion_y_componentes_logistica.png': 'solucion_y_componentes_logistica.png',
    'solucion_y_componentes_poisson_seno.png': 'solucion_y_componentes_poisson_seno.png',
    'solucion_y_componentes_poisson_tanh_con_datos.png': 'solucion_y_componentes_poisson_tanh_con_datos.png',
    'solucion_y_componentes_poisson_tanh_sin_datos.png': 'solucion_y_componentes_poisson_tanh_sin_datos.png',
    'desglose_perdidas_logistica.png': 'desglose_perdidas_logistica.png',
    'desglose_perdidas_poisson_seno.png': 'desglose_perdidas_poisson_seno.png',
    'desglose_perdidas_poisson_tanh_con_datos.png': 'desglose_perdidas_poisson_tanh_con_datos.png',
    'desglose_perdidas_poisson_tanh_sin_datos.png': 'desglose_perdidas_poisson_tanh_sin_datos.png',
    'desglose_perdidas_regularizacion.png': 'desglose_perdidas_regularizacion.png',
    'participacion_perdidas_poisson_2d_con_datos.png': 'participacion_perdidas_poisson_2d_con_datos.png',
    'participacion_perdidas_poisson_2d_sin_datos.png': 'participacion_perdidas_poisson_2d_sin_datos.png',
    'logistica_edo.png': 'comparison.png',
    'perdida_poisson_seno.png': 'convergence.png',
    'poisson_1d_tanh_con_datos.png': 'comparison3_data.png',
    'poisson_1d_tanh_sin_datos.png': 'comparison4_wdata.png',
    'poisson_2d_con_datos.png': 'comparison6.png',
    # Comparaciones originales ampliadas y cuantificadas.
    'figura_4_7_regularizacion.png': 'figura_pinn_media_lipr_1d.png',
    'comparacion_3d_soluciones_errores_Poisson2D.png': 'comparacion_3d_soluciones_errores_poisson2d.png',
    'figura_4_9_comparacion_metodos.png': 'comparacion_fem_fvm_pinn_reproducida.png',
    'errores_espaciales_FEM_FVM_PINN.png': 'mapas_errores_fem_fvm_pinn.png',
    # Figuras nuevas: ya se guardaron con el nombre esperado en el .tex.
    'reproduccion_shin_figura3.png': 'reproduccion_shin_figura3.png',
    'reproduccion_shin_figura4.png': 'reproduccion_shin_figura4.png',
    'diagnostico_holder_poisson1d.png': 'diagnostico_holder_poisson1d.png',
}
carpeta_latex = SALIDA/'imagenes'
carpeta_latex.mkdir(exist_ok=True)
registro=[]
for original,destino in nombres_figura.items():
    ruta=next((carpeta/original for carpeta in RUTAS_FIGURAS if (carpeta/original).exists()), SALIDA/original)
    assert ruta.exists(), f'Falta la figura requerida: {ruta}. Revise la ejecución.'
    shutil.copy2(ruta,carpeta_latex/destino)
    registro.append({'origen':original,'archivo_latex':f'imagenes/{destino}',
                     'bytes':(carpeta_latex/destino).stat().st_size})
registro_figuras=guardar_csv('indice_figuras_capitulo4.csv',registro)
print(registro_figuras.to_string(index=False))
print('Figuras listas para el capítulo:',carpeta_latex)


## Resultados y referencias

Las curvas y superficies numéricas se calculan durante la ejecución. La carpeta `resultados_pinn` (o la carpeta elegida en Google Drive) contiene las figuras, tablas CSV y los parámetros del experimento. Las gráficas 3D y los mapas espaciales comparten escalas para comparar errores puntuales. La comparación debe interpretarse según el número de puntos de colocación, las arquitecturas y los hiperparámetros indicados; no constituye una demostración numérica general de los teoremas asintóticos.

**Referencias principales:**

- Raissi, M., Perdikaris, P. y Karniadakis, G. E. (2019). *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations*. Journal of Computational Physics, 378, 686–707.
- Shin, Y., Darbon, J. y Karniadakis, G. E. (2020). *On the Convergence of Physics Informed Neural Networks for Linear Second-Order Elliptic and Parabolic Type PDEs*. Communications in Computational Physics, 28(5), 2042–2074. https://doi.org/10.4208/cicp.OA-2020-0193
- Ilustraciones conceptuales de introducción: [Physics-Informed Neural Networks tutorial](https://github.com/FilippoMB/Physics-Informed-Neural-Networks-tutorial), de Filippo Maria Bianchi y colaboradores (enlaces a imágenes indicados junto a cada figura).

In [ ]:
# Comprimir los resultados, los metadatos y la carpeta imagenes/ del capítulo.
import zipfile
archivos = sorted(p for p in SALIDA.rglob('*') if p.is_file())
print('Archivos disponibles:',len(archivos))
for archivo in archivos:
    print(' -',archivo.relative_to(SALIDA))
archivo_zip=SALIDA.parent/('Resultados_Capitulo_IV_'+MODO+'_compilado.zip')
with zipfile.ZipFile(archivo_zip,'w',zipfile.ZIP_DEFLATED) as zf:
    for archivo in archivos:
        zf.write(archivo,arcname=str(archivo.relative_to(SALIDA)))
print('Resultados y carpeta imagenes/ comprimidos en:',archivo_zip)
